# Assignment 05 - Engineering our pipeline

### Due: Monday, Feb 23, before class

Name(s): Chang Min Bark<br>
Course: CSCI 357 - AI and Neural Networks<br>
Section: 01 - 1:00pm<br>
Semester: Spring 2026<br>
Instructor: Prof. King<br>

### References
* PyTorch Documentation: https://pytorch.org/docs/stable/index.html
* Weights & Biases Documentation: https://docs.wandb.ai/
* ...and a ton of Google and various AI interaction

In [1]:
import sys
import os
import time
from enum import Enum

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import seaborn as sns

# Basic configuration
torch.manual_seed(42)
np.random.seed(42)
sns.set_theme(style="whitegrid")

print(f"PyTorch version: {torch.__version__}")

PyTorch version: 2.9.0+cu128


In [2]:
from IPython.display import HTML, display

# Define CSS styles in a string variable for reuse across cells
css_styles = """
<style>
.todo-box {
    background-color: #fcf8e3;
    color: #000000;
    border-left: 6px solid #f0ad4e;
    padding: 10px;
    margin-bottom: 0;
}

.todo-box h3 {
    margin-top: 0;
}

.note-box {
    background-color: #d1ecf1;
    color: #000000;
    border-left: 6px solid #0c5460;
    padding: 10px;
    margin-bottom: 0;
}

.warning-box {
    background-color: #f8d7da;
    color: #000000;
    border-left: 6px solid #721c24;
    padding: 10px;
    margin-bottom: 0;
}

.section-box {
    background-color: rgb(125, 233, 150);
    color: #000000;
    border-left: 6px solid #155724;
    padding: 10px;
    margin-bottom: 0;
}
</style>
"""

display(HTML(css_styles))

In [3]:

def show_todo(todo_text):
    display(HTML(f"""
    <div class="todo-box">
    <h2>TODO: {todo_text}</h2>
    </div>
    """))

def show_note(note_text):
    display(HTML(f"""
    <div class="note-box">
    <h2>NOTE:</h2>
    <h4>{note_text}</h4>
    </div>
    """))

def show_warning(warning_text):
    display(HTML(f"""
    <div class="warning-box">
    <h2>WARNING:</h2>
    <h3>{warning_text}</h3>
    </div>
    """))

def show_section(section_text = ""):
    display(HTML(f"""
    <div class="section-box">
    <h1>{section_text}</h1>
    </div>
    """))

In [4]:
show_todo("Set Run Configuration")

# **Notebook Run Configuration**


As most of you have already discovered, you will frequently need to re-run your notebooks from scratch. As our models and data become larger and more complex, code cells that execute training can take a lot of time. Thus, once you have finish a cell that needs to train, you might want to just execute a few epochs, or perhaps NO epochs.

* Set the variable `RUN_TRAINING_MODE` below to one of the three enumerations, then write your code to conditionally set the number of epoch to run
* Set `DO_WANDB_LOGGING` to True when you are ready to log your results.

It's up to you to properly use these constants in your code to make it easy to rerun a notebook from scratch.

In [5]:
class RunTrainingMode(Enum):
    NONE = "NONE"
    QUICK = "QUICK"
    FULL = "FULL"

RUN_TRAINING_MODE = RunTrainingMode.FULL
DO_WANDB_LOGGING = True

# **Set up accelerator device**
> This code will create three variables
> * `cpu_device` - the default device all tensors are stored on when not explicity moving over to a GPU
> * `has_accel` - bool - `True` if you have a GPU, `False` otherwise.
> * `accel_device` - your accelerator device if you have a GPU, or set to `cpu_device` if no GPU is present to prevent exceptions if trying to run on a GPU when one is not available.


In [6]:
cpu_device = torch.device("cpu")

# True if *some* accelerator is available (CUDA, MPS, etc.)
has_accel = torch.accelerator.is_available()
print(f"Accelerator available: {has_accel}")

# A torch.device representing the preferred accelerator (if any). If none available, set it to "cpu"
if has_accel:
    accel_device = torch.accelerator.current_accelerator()
    print(f"Preferred accelerator: {accel_device}")
else:
    accel_device = cpu_device
    print("No accelerator available, defaulting to CPU.")

Accelerator available: True
Preferred accelerator: cuda


In [7]:
# Are we running from colab?
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab")
else:
    print("Not running in Google Colab")

Running in Google Colab


In [8]:
show_todo("Set Project Path and Install Packages")

**TODO:** You need to set `project_path` to be the root path of your project. This needs to be done regardless if whether you are running from Colab or not. I included an example that works on my machine when running locally, and setting a Drive path when running from Google Colab.

In [9]:
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print("Drive mounted successfully!")

    # DONE: Replace this with your Google Drive path to your project root. Below is an example:
    PROJECT_PATH = '/content/drive/My Drive/2026 SS/CSCI357/csci357_2026sp'

    # Confirm the project_path exists, raise an error if not
    if not os.path.exists(PROJECT_PATH):
        raise FileNotFoundError(f"Project path NOT FOUND: \"{PROJECT_PATH}\"")
    else:
        print(f"Confirmed: Project path exists at \"{PROJECT_PATH}\"")
else:
    # DONE: Replace this with your project root.
    PROJECT_PATH = '/content/drive/My Drive/2026 SS/CSCI357/csci357_2026sp'

print(f"Project path: \"{PROJECT_PATH}\"")
if PROJECT_PATH not in sys.path:
    sys.path.append(PROJECT_PATH)
    sys.path.append(PROJECT_PATH + '/src')

# Ensure that the current working directory is PROJECT_PATH
if os.getcwd() != PROJECT_PATH:
    print(f"Changing working directory to project path: \"{PROJECT_PATH}\"")
    os.chdir(PROJECT_PATH)
else:
    print(f"Already in the correct working directory: \"{PROJECT_PATH}\"")

Mounted at /content/drive
Drive mounted successfully!
Confirmed: Project path exists at "/content/drive/My Drive/2026 SS/CSCI357/csci357_2026sp"
Project path: "/content/drive/My Drive/2026 SS/CSCI357/csci357_2026sp"
Changing working directory to project path: "/content/drive/My Drive/2026 SS/CSCI357/csci357_2026sp"


In [10]:
show_warning("Colab Users: THIS NEXT CELL WILL NOT WORK UNTIL YOU COMPLETE SECTION 1 BELOW!")

In [11]:
# Install any packages that are not part of the standard Google Colab environment
if IN_COLAB:
    # Install uv for fast installs (-q for quiet, -U for update)
    !pip install -qU uv
    # Install project dependencies and local code in editable mode,
    !uv pip install --system -e .
    # !uv pip show scipy
    # !uv pip show torchmetrics
    !uv pip list
    pass
else:
    print("Running locally, assuming project dependencies installed in your local environment.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 68.4 MB/s eta 0:00:00:00:0100:01
Using Python 3.12.12 environment at: /usr
Resolved 198 packages in 749ms                                       
Prepared 21 packages in 4.58s                                            
Uninstalled 3 packages in 29ms
Installed 21 packages in 185ms                              
 + asttokens==3.0.1
 + async-lru==2.1.0
 - cachetools==7.0.1
 + cachetools==6.2.6
 + executing==2.2.1
 - ipython==7.34.0
 + ipython==9.10.0
 + ipython-pygments-lexers==1.1.1
 + jedi==0.19.2
 + json5==0.13.0
 + jupyter==1.1.1
 + jupyter-lsp==2.3.0
 + jupyterlab==4.5.4
 + jupyterlab-server==2.28.0
 + lightning-utilities==0.15.2
 + my-engine==0.1.0 (from file:///content/drive/My%20Drive/2026%20SS/CSCI357/csci357_2026sp)
 + pure-eval==0.2.3
 + pydeck==0.9.1
 + stack-data==0.6.3
 + streamlit==1.54.0
 + torchmetrics==1.8.2
 - traitlets==5.7.1
 + traitlets==5.14.3
 + ucimlrepo==0.0.7
Using Python 3.12.12 environment at: /usr
Packag

Once you work through the initial setup of `pyproject.toml` then you can uncomment the two lines above, and the one line below. Colab users have a minor inconvenience with environment setup each time, but a small price to pay for access to fantasic GPUs.

In [12]:
# Once packages are installed, we can kill the kernel to force a restart. Just restart the kernel
if IN_COLAB:
    try:
        import my_engine
        print(my_engine.__file__)
        print("Success!")
    except Exception as e:
        print("Triggering restart of kernel.")
        # This will force the kernel to restart
        import os
        # os.kill(os.getpid(), 9)


/content/drive/My Drive/2026 SS/CSCI357/csci357_2026sp/src/my_engine/__init__.py
Success!


In [13]:
# This cell loads the 'autoreload' extension for IPython/Jupyter, which automatically reloads any modules
# you import whenever their code changes on disk. The first line loads the extension, and the second line
# sets autoreload to mode 2, which reloads all modules (except those excluded by %aimport) before each execution.
# This is especially useful when you're actively developing Python modules in separate files and want
# changes to be picked up immediately without restarting the notebook kernel.

# Turn on the "Auto-Updater"
%load_ext autoreload
%autoreload 2

In [14]:
import wandb

In [15]:
show_todo("Set W&B Configuration Variables")

# Set your identity variables for W&B!

Fill in the following variables below to ensure your data is properly logged to Weights & Biases

In [16]:
# DO NOT CHANGE THIS:
entity = "bucknell-university-csci357-2026sp"

# DONE: Fill in your name, user id, and email
user_name = "Chang Min Bark"
user_initials = "CMB"
user_id = "cb073"
user_email = "cb073@bucknell.edu"

In [17]:
show_section()

---
# Background




We're just about done with the foundations of the multilayer perceptron and deep feedforward architectures. Think about the size of that .ipynb file in the previous assignment! It was nuts! Your notebooks are getting bigger and bigger! You must be thinking...

> *Is King nuts? This is not sustainable! He can't keep forcing these multi-megabyte notebooks on us! There must be a better way!*

Yes, you would be right. No, not that I'm nuts, but that there is a better way!

Because this course is emphasizing **MLOps**, best practices followed by the pros working in this field, we're now ready to rethink our project structures. We can't possibly stick to our current approach as we learn more challenging, complex topics. Look at the chaos we created in the last notebook! It was a mess! This approach is not scalable. Notebooks like these are difficult to manage and find the information you need. They are enormous. Slow to load. Unweildy. And consider the code itself. You must have noticed that for the challenges, you were following the same **pattern**: create the model, criterion, optimizer, set up loop over lots of hyperparameters, and GO! Anytime you see a type of design or implementation pattern like this, it should be screaming to you to modularize and channel some better OPPnot reproducible. You could not replicate the results you shared on wandb even if you tried (unless you were wise enough to reset your random number seed.)

**It's time to move from our "monolithic notebooks" to a more modular approach.**

### Why are we doing this?

Your notebooks have been growing larger and more unwieldy. This is the natural evolution from experimentation to production-ready code. In industry, notebooks are for experimentation, but Python packages are for production. By modularizing your code now, you make it:
✅ Testable - Individual components can be unit tested
✅ Reusable - Functions work across multiple experiments
✅ Version-controllable - Git tracks meaningful changes to logic, not notebook output noise
✅ Scalable - Easy to extend and collaborate on
✅ Maintainable - Clean separation of concerns

---

**Example: A well-structured AI/ML engineer's project space**<br>
Look at the following structure and think about how you would reorganize your project folder to match it. Most of this makes sense, though we're not really at the point where we need this much structure yet. But, this is the general consensus of MLops practitioners (grabbed from [this reference](https://towardsdatascience.com/structuring-your-machine-learning-project-with-mlops-in-mind-41a8d65987c9/):

```
my_project/
├── README.md
├── pyproject.toml / setup.cfg / requirements.txt
├── configs/
│   ├── data.yaml
│   └── model_xgboost.yaml
├── data/
│   ├── raw/
│   ├── interim/
│   └── processed/
├── notebooks/
│   ├── 01_exploration.ipynb
│   └── 02_error_analysis.ipynb
├── src/  # or my_project/
│   ├── __init__.py
│   ├── data/
│   │   ├── load.py
│   │   └── preprocess.py
│   ├── features/
│   │   └── build_features.py
│   ├── models/
│   │   ├── train.py
│   │   └── evaluate.py
│   └── utils/
│       └── io.py
├── scripts/
│   ├── train.py
│   └── evaluate.py
└── tests/
    ├── test_preprocess.py
    └── test_models.py
```

> References: https://towardsdatascience.com/structuring-your-machine-learning-project-with-mlops-in-mind-41a8d65987c9/

In [18]:
show_section()

# Lab

## Section 1: Restructuring our Project

We have a lof of work ahead for this week. It is criticial that you take your time working through this. Understand each step. Do not rush. The better job you do with this week's work, the more enjoyable time you will have in the future weeks ahead as we get into more complex networks structures.


In [19]:
show_note("I want to strongly encourage you to take your time with this. Reading up on best practices for ML and AI engineering formed the basis of this assignment. It is critical that you understand each step and why we are doing it. If there was ever a good time to use AI for assistance with your work, this is it!")
show_warning("BE CAREFUL WITH AI ASSISTANCE! I switched between the best AI engines as I tested out this project several times, and I literally lost count of how many times it actually slowed me down because it injected too many bugs. Most of the time, the problem was because I was either too vague with my instructions, or allowed my chat to get too long. Yes, I want you to use it, and be careful while doing so.")


### Step 1: Prepare your new project structure

* In your project folder for the course, create the following top-level folders: `data`, `docs`, `models`, `notebooks`, `reports`, `src`, and `tests`
* In your `src` folder:
    * Create a folder called `my_engine`. We'll use this folder to store all our custom, reusable PyTorch code. This will be the folder we'll keep our highest quality, reusable code in. This is code we will reuse for much of the semester.
* In your `src/my_engine` folder:
    * Create an empty file called `__init__.py`. This is a special file that tells Python that this folder should be treated as a Python package.
* Move all your `.ipynb` notebooks into the `notebooks` folder.
    * Personally, I keep each week's notebook in a separate folder under this folder, labeled `01`, `02`, etc. because I have more than just a single notebook per week. I have several markdown files to keep my own notes, AI prompts, ideas, code snippets, and other relevant information that I collect for every week. You are not required to do this. Just an idea.

### Step 2: Add a `pyproject.toml` to Your Project

To set up your project with modern Python practices, simply do the following:

1. Set up a new empty file in your **root directory of your project**. Name it `pyproject.toml`.
2. **Copy the template below and paste it into your `pyproject.toml` file.**
3. Verify that the name field is `my_engine` and that you have a folder `src/my_engine` in your project. Optionally, update the `name` field to match your project.
   - The `name` field sets the importable Python package name. For example, if you put `name = "my_engine"`, then after installation you can `import my_engine` in your Python code. This does **not** create or rename any folders, but determines how others will import your code after running `pip install -e .`.

That's it! This enables dependency management, clean installs, and a professional project structure.

`pyproject.toml` template for getting started<br><br>
<br>
```
[build-system]
requires = ["setuptools>=61.0"]
build-backend = "setuptools.build_meta"

[project]
name = "my_engine"  # <-- this is your importable package name. It needs to match the name of the folder you created under src/.
version = "0.1.0"
description = "Student project for CSCI 357 Coursework"
dependencies = [
    "jupyter", "jupyterlab", "jupytext", "nbconvert",  "ipython", "ipykernel",
    "numpy", "pandas", "matplotlib",
    "scikit-learn", "statsmodels",
    "torch", "torchvision", "torchaudio", "torchmetrics",
    "fastapi", "uvicorn", "gradio", "streamlit",
    "wandb", "tqdm",
    "kaggle", "kagglehub", "ucimlrepo"
]

[tool.setuptools.packages.find]
where = ["src"]
```

---
<br>

### Step 3: Connecting your project to your new `pyproject.toml`

There are two main ways to set up your Python environment, depending on where you run your code:

-----
#### **A. Local Environment (Conda + uv)**

If you are running code on your own machine with Conda:

1. Open a terminal and make sure you have activated your Conda environment for this course.
2. Navigate to the root folder of your project (the one containing `pyproject.toml`).
3. Run:
   ```
   uv pip install -e .
   ```
   This will install/update your project and all its dependencies in "editable" mode.

**Important:** If you add a new library to your `pyproject.toml`, just run `uv pip install -e .` again locally (or restart your Colab cell). It will only install the new addition!
You only need to run this once after you create your project structure and add your `pyproject.toml` file.

-----
#### **B. Google Colab Users**

If you use Google Colab, you'll notice that we improved the setup at the beginning of this notebook to already add the project path to your Python search path.

Assuming you did that step, AND you have added the `pyproject.toml` file to your project root, you can simply run the previous import cell to install your dependencies and your package.

You'll notice that this is also included in the initialization cells at the beginning of this notebook. Now, anytime you add a new library to your `pyproject.toml`, just re-run the appropriate install command (either in your local terminal or re-run the Colab cell) to update your environment.


> **Pro-Tip:** If you add a new library (like `scipy`) to your `pyproject.toml`, just run `uv pip install -e .` again loclly (or restart your Colab cell). It will only install the new addition!



---
## Section 2: REFACTORING FOR ML SUCCESS


### Where we left off (HW04 recap)

Last week you built the core pieces of training, but you did it *inside the notebook* over and over:

- Wrote `train_epoch_minibatch()` and `evaluate_minibatch()` once, then copied the loop for each experiment.
- Repeated the "magic 5 steps" (zero grad, forward, loss, backward, step) in many cells.
- Repeated W&B `init -> log -> finish` blocks for every experiment.
- Implemented early stopping as a separate class, then stitched it into a training loop by hand.
- Re-selected devices and re-wired DataLoaders every time.
- Created chaotic hyperparameter sweeps for each experiment, with deep nested loops.
- Created a new model class for each experiment, using `create_new_mlp()` function.

That chaos was not quite intentional, as this is pretty much how it goes when you are first learning any AI/ML framework. However, it's chaos, and not scalable. These are classic "code smells"! We needed to *feel* the pain of duplication, redundancy, and lack of modularity! **And it's time to stop the chaos!**<br>

This week we eliminate our spaghetti code by creating reusable, scalable, and modular code. This is essential to building a robust framework that will make the remainder of the semester much more enjoyable and productive! By the end of this assignment, you're going to have your own "engine" that you can reuse for the remainder of this semester and beyond. Moreoever, once again, this is MLOps in action! This is how it's done by the pros. Creating chaotic notebooks is very much viewed as something "noobs" do, and represent clear signs that you are just learning the basics. It's time to smell like a well-seasoned pros!

**Key shift:** Treat training as a long-running process, not a one-off script. That means we need:
- A *stateful* Trainer that can pause, resume, and recover after a crash.
- A configuration class for managing hyperparameters and configuration.
- A *single* place for training/validation logic.
- Internal logging (including W&B).
- Built-in early stopping.
- Checkpointing and resuming training from a checkpoint (essential for long training runs and experimentation)

This is exactly how professional ML systems are built. It's not just about the code, and thanks to AI, never has this been more true for the software engineer. It's about the process and the principles.

Let's get started!


#### Step 2.1 - Encapsulating model and training configuration and hyperparameters with a config classes

First, let's deal with figuring out a way to manage our configuration and hyperparameters. Let's create a `TrainerConfig` class that will store the configuration for the `Trainer`, and a `ModelConfig` class that will store the configuration for the `Model`. We'll make use of the `dataclasses` module to create a class that will store the configuration for the `Trainer` and `Model`.<br>

`dataclasses` is a built-in Python module. It was introduced back in Python 3.7, but not utilized enough. It's a very useful module that automatically generates boilerplate code for classes that primarily store data... such as storing the configuration and hyperparameters for the `Trainer`!

Let's clarify what `dataclasses` is so you can not only understand it, but get a good appreciation for these utility classes. It's basically a shortcut to avoid writing repetitive `__init__`, `__repr__`, and `__eq__` methods. Let's see an example of what it would look like without and with `dataclasses`:

**The "Old" Way (Without dataclasses)**

```python
class TrainerConfig:
    def __init__(self, learning_rate=1e-3, batch_size=32, device="cpu"):
        self.learning_rate = learning_rate
        self.batch_size = batch_size
        self.device = device

    def __repr__(self):
        return f"TrainerConfig(lr={self.learning_rate}, batch={self.batch_size}, dev={self.device})"

    def __eq__(self, other):
        return (self.learning_rate == other.learning_rate and
                self.batch_size == other.batch_size and
                self.device == other.device)
```

Notice how `self.variable = variable` is repeated over and over? Silliness. *I know this is an AI/ML course, but if you don't pay attention to understanding the underlying code, even when it's generated for you, you're not going to get very far!* When you see repetitive patterns in code, there's a good chance you can refactor it to be more efficient, or more likely, the backbone frameworks and API in the language itself probably has a solution to help you out.<br>

**The "New" Way (With dataclasses)**

The `@dataclass` decorator writes all that code for you behind the scenes:

```python
from dataclasses import dataclass

@dataclass
class TrainerConfig:
    learning_rate: float = 1e-3
    batch_size: int = 32
    device: str = "cpu"
```

That's it!

When you use `@dataclass`, Python automatically adds:
1.  **`__init__`**: So you can do `config = TrainerConfig(batch_size=64)`.
2.  **`__repr__`**: So `print(config)` gives you a nice readable string: `TrainerConfig(learning_rate=0.001, batch_size=32, device='cpu')`.
3.  **`__eq__`**: So you can compare two configs: `config1 == config2` works as expected.

`dataclass` is perfect for managing configurations and hyperparameters for your models. It's a great way to keep your code clean and organized:
1.  **Readability**: You see the fields and their types immediately.
2.  **Less Code**: No massive `__init__` method cluttering the file.
3.  **Type Hints**: It forces you to use type hints (`: int`, `: str`), which helps your editor catch bugs.
4.  **`asdict`**: It comes with a helper function `dataclasses.asdict(obj)` that converts your class instance back into a regular dictionary (useful for saving to JSON or passing to libraries that expect dicts) (such as W&B!)
5. It does not allow other entries to be added to the config class, which is a good thing!

> **Pro-Tip:** Use `dataclasses.asdict(config)` to convert the config class to a dictionary.

Let's create a `TrainerConfig` and `ModelConfig` classes. We're going to make use of the `dataclasses` module to create a class that will store the configuration for the `Trainer` and `Model` respectively.<br>



In [ ]:
if os.path.exists("src/my_engine/config.py"):
    print("Warning: src/my_engine/config.py already exists. Not overwriting.")
else:
    # Use IPython magic to write only if file doesn't exist
    from IPython import get_ipython
    ipython = get_ipython()
    ipython.run_cell_magic("writefile", "src/my_engine/config.py", """
\"\"\"
config.py

This module contains configuration dataclasses useful for AI and ML projects.

Course: CSCI 357 - AI and Neural Networks
Author: [Your Name]
Date: [Current Date]

\"\"\"

from dataclasses import dataclass, field
import torch
from typing import List, Optional

@dataclass
class TrainerConfig:
    trainer_batch_size: int = 64
    evaluator_batch_size: int = 256
    learning_rate: float = 0.001
    device: torch.device = torch.device("cpu")
    num_epochs: int = 10
    weight_decay: float = 0.0
    early_stopping_patience: Optional[int] = 5          # Set to None to disable early stopping
    early_stopping_min_delta: float = 0.001

@dataclass
class ModelConfig:
    model_type: str = "mlp"
    hidden_units: List[int] = field(default_factory=lambda: [128, 64])
    dropout: List[float] = field(default_factory=lambda: [0.1, 0.2])

""")

Notice those classes pretty much encapsulate the vast majority of our configuration and hyperparameters we used in the previous assignment. We're following good practices of separation of training and model parameters. This is a good practice to follow in your own code. We're also using the `field` function to initialize the fields with default values. This is a good practice to follow in your own code.

> **Pro-Tip:** Use `field(default_factory=lambda: [128, 64])` to initialize the fields with a default factory function. This is a good practice to follow in your own code. Don't just accept the code I give you! Dive in! Make sure you understand it!
>
> AGAIN - another great use case for AI tools for the software and AI engineer!
>
> *"Explain this line of code to me. What is the field function? What is the lambda function? What is the default_factory function? Help me understand it in a general sense so I can reuse these techniques in other projects."*

Let's test out where we're at so far. Let's create a `TrainerConfig` and `ModelConfig` instance and print it out.

In [21]:
from my_engine.config import TrainerConfig, ModelConfig
from dataclasses import asdict

trainer_config = TrainerConfig(trainer_batch_size=64, evaluator_batch_size=256, learning_rate=0.001, device=torch.device("cpu"))
model_config = ModelConfig(model_type="mlp", hidden_units=[128, 64], dropout=[0.1, 0.2])

print(trainer_config)
print(asdict(trainer_config))
print(model_config)
print(asdict(model_config))

TrainerConfig(trainer_batch_size=64, evaluator_batch_size=256, learning_rate=0.001, device=device(type='cpu'), num_epochs=10, weight_decay=0.0, early_stopping_patience=5, early_stopping_min_delta=0.001, optimizer_name='adam', momentum=0.9, checkpoint_dir='./checkpoints', checkpoint_last_filename='last.pt', checkpoint_save_interval=5, checkpoint_best_filename='best.pt', use_scheduler=False, scheduler_type='reduce_on_plateau', scheduler_step_size=10, scheduler_gamma=0.1, scheduler_patience=3, scheduler_min_lr=1e-06, num_classes=10)
{'trainer_batch_size': 64, 'evaluator_batch_size': 256, 'learning_rate': 0.001, 'device': device(type='cpu'), 'num_epochs': 10, 'weight_decay': 0.0, 'early_stopping_patience': 5, 'early_stopping_min_delta': 0.001, 'optimizer_name': 'adam', 'momentum': 0.9, 'checkpoint_dir': './checkpoints', 'checkpoint_last_filename': 'last.pt', 'checkpoint_save_interval': 5, 'checkpoint_best_filename': 'best.pt', 'use_scheduler': False, 'scheduler_type': 'reduce_on_plateau', 

Notice how the `hidden_units` and `dropout` fields are initialized with a default factory function? Don't just accept the code I give you! Dive in! Make sure you understand it!

Assuming you follow all of the instructions for reorganizing your project, creating the src directory, creating the my_engine package, and creating the config.py file, you should have now had the my_engine package available in your own code, ready for you to import and use. Hopefully are you now starting to appreciate the importance of modularity and separation of concerns in your own code. We're only getting started!



#### Step 2.2 - Write a `MLP_Model` class


For this exercise, you are going to take the lead. I'll give you a starter version of the `MLP_Model` class. You'll need to fill in the details.

Start by running the following cell to create the initial file structure for the `MLP_Model` class in `src/my_engine/model.py`. You'll need to fill in the details. As usual, look for all the **TODO:** markers.

> NOTE: Notice how these cells use IPython magic called writefile, but we're using the `IPython` interface to run the cell magic. This gives us some extra ability to ensure we conditionally write the files only if they don't already exist. This is a good practice to follow in your own code. That way, you don't overwrite the file if it already exists (thus allowing you to run the notebook again.)


In [22]:
if os.path.exists("src/my_engine/model.py"):
    print("Warning: src/my_engine/model.py already exists. Not overwriting.")
else:
    from IPython import get_ipython
    ipython = get_ipython()
    ipython.run_cell_magic("writefile", "src/my_engine/model.py", """
import torch
import torch.nn as nn
from src.my_engine.config import ModelConfig

class MLP_Model(nn.Module):
    def __init__(self, num_inputs: int, num_outputs: int, config: ModelConfig):
        super().__init__()
        # TODO: Implement the initialization of the MLP model.
        pass

    def forward(self, x):
        # TODO: Implement the forward pass of the MLP model.
        pass

    def num_parameters(self) -> tuple[int, int]:
        # TODO: Implement the number of parameters of the MLP model.
        pass

    def __str__(self) -> str:
        # TODO: Implement the string representation of the MLP model.
        pass

    def __repr__(self) -> str:
        # TODO: Implement the representation of the MLP model.
        pass
""")

show_todo("Finish the MLP_Model class")

**TODO:** Finish each of the above. If you're lost, check out the function that constructed MLP models from the previous assignment. That's essentially what we're trying to do.

> Remember, the model class will ONLY encapsulate the model. There is NO training or validation logic in this class!
>
> We're trying to minimize coupling betwen our classes and maximize cohesion within each class. This is a key principle of good software engineering and design.

Let's instantiate our first model, print it out, and make sure we've got it working. We'll use our `model_config` from above to create the model.

In [23]:
from my_engine.model import MLP_Model
model = MLP_Model(num_inputs=784, num_outputs=10, config=model_config)
print(model)
print(model.num_parameters())

MLP_Model(num_inputs=784, num_outputs=10, config=ModelConfig(model_type=<ModelType.MLP: 'mlp'>, hidden_units=[128, 64], dropout=[0.1, 0.2]))
(109386, 109386)


If you did it right, you should see this type of result:

```
MLP_Model(num_inputs=784, num_outputs=10, config=ModelConfig(model_type='mlp', hidden_units=[128, 64], dropout=[0.1, 0.2]))
(109386, 109386)
```

#### Step 2.3 - Create a `utils.py` module

Good software engineering in Python expects that you create well-defined packages with modules. It's about separating responsibilities, unlike what we've been doing - shoving ALL responsibilities in one notebook!<br><br>

Let's create a `utils.py` module. It's going to serve the purpose of containing helper functions for our experiments. For now, let's keep it simple. It will contain the `accuracy_from_logits` function we used in the previous assignment. We'll also create a `build_model` function that will build model based on the configuration.

Execute the cell below. It will write your new utils.py file to get you started. Then go to the file and complete both functions.

> HINT: Look for the `create_new_mlp` function in your previous homework. It'll have a lot of overlapping functionality.  

In [24]:
if os.path.exists("src/my_engine/utils.py"):
    print("Warning: src/my_engine/utils.py already exists. Not overwriting.")
else:
    # Use IPython magic to write only if file doesn't exist
    from IPython import get_ipython
    ipython = get_ipython()
    ipython.run_cell_magic("writefile", "src/my_engine/utils.py", """
\"\"\"
utils.py

This module contains a collection of helper utility functions that will be used throughout the course.
You will find reusable functions for metrics, data handling, and other tools to support labs,
assignments, and projects.

Course: CSCI 357 - AI and Neural Networks
Author: [Your Name]
Date: [Current Date]

\"\"\"

def accuracy_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:
    # TODO: Implement the accuracy_from_logits function.
    pass

def build_model(num_inputs: int, num_outputs: int, config: ModelConfig) -> nn.Module:
    # TODO: Implement the build_model function.
    pass
""")

In [25]:
show_todo("Finish the utils.py module")

Let's test out our `build_model` function and the `accuracy_from_logits` function.

In [26]:
from my_engine.utils import accuracy_from_logits, build_model, make_optimizer

print("Building model with config: ", model_config)
model = build_model(784, 10, model_config)
print(model)
print(model.num_parameters())

print("Simple test for accuracy_from_logits function:")
import torch

# Hard-coded logits: each row corresponds to a sample (10 samples, 10 classes).
# We'll make it so class i is the maximum in row i, so accuracy should be 1.0
logits = torch.eye(10) * 10  # 10x10 tensor, diagonal elements are 10 (others 0)
logits[[0, 1]] = logits[[1, 0]]

# Labels: The correct class is i for sample i (i.e., torch.arange(10))
labels = torch.arange(10)

acc = accuracy_from_logits(logits, labels)
print(f"Expected accuracy is 0.8 got {acc:.2f}")

Building model with config:  ModelConfig(model_type=<ModelType.MLP: 'mlp'>, hidden_units=[128, 64], dropout=[0.1, 0.2])
MLP_Model(num_inputs=784, num_outputs=10, config=ModelConfig(model_type=<ModelType.MLP: 'mlp'>, hidden_units=[128, 64], dropout=[0.1, 0.2]))
(109386, 109386)
Simple test for accuracy_from_logits function:
Expected accuracy is 0.8 got 0.80


#### Step 2.4 - Write the `Trainer` module

In the last assigment, you built the core loop functions (train_epoch_minibatch, evaluate_minibatch) and used them everywhere, but each experiment still rewired the same steps. You hardcoded multiple training sweeps (optimizer comparison, regularization, early stopping) repeated device setup, dataloaders, W&B init/log/finish, etc. Then there was early stopping as a separate helper (EarlyStopper) and was manually stitched into each loop. On top of it all was repeated loop glue. Yeah, that's right... I said it - **LOOP GLUE**.

![A tired BRK from dealing with ridiculously long, unmanageable notebook files](https://www.eg.bucknell.edu/~brk009/wp-content/uploads/2026/02/BRK_loop_glue.png)

> *LOOP GLUE: /lo͞op ɡlo͞o/ (noun) – The mysterious, sticky substance found everywhere in ML notebooks, responsible for holding together one-off training scripts, copy-pasted cell experiments, and multiple snippets and chunks of code whose only job is to reset random seeds, call `.to(device)` over and over. It often amounts to crap code that's highly flammable, severely under-commented, and prone to attracting bugs.*

It was chaos from a software engineering perspective. So much duplication and so much code that is hard to maintain and update, which made the notebook huge and fragile. Now, we're going to build a `Trainer` class that will encapsulate all of this logic.

Our `Trainer` class will encapsulate everything you need to manage training a new model. We'll have quite aq bit to implement, but let's get you started. We'll implement the following methods:
- `train_one_epoch()`
- `validate()`
- `fit()`

**TODO:** Execute the next cell. It's a starter version of the `Trainer` class. This is going to be one heck of an important class for the remainder of the semester, so make sure you understand it well. The `Trainer`` class has NO comments provided. This is intentionally done to force you to read the code and understand it. Use AI to break it down and document it.

In [27]:
if os.path.exists("src/my_engine/trainer.py"):
    print("Warning: src/my_engine/trainer.py already exists. Not overwriting.")
else:
    from IPython import get_ipython
    ipython = get_ipython()
    ipython.run_cell_magic("writefile", "src/my_engine/trainer.py", """
import torch
import torch.nn as nn
import torch.optim as optim
import wandb
from typing import Callable, Tuple, Optional, Dict
from dataclasses import dataclass

from my_engine.config import TrainerConfig, ModelConfig
from my_engine.utils import accuracy_from_logits

class Trainer:
    def __init__(
        self,
        model: nn.Module,
        optimizer: optim.Optimizer,
        criterion: Callable[[torch.Tensor, torch.Tensor], torch.Tensor],
        config: TrainerConfig = TrainerConfig(),
        run: Optional[wandb.Run] = None,
    ) -> None:
        self.model = model.to(config.device)
        self.optimizer = optimizer
        self.criterion = criterion
        self.config = config
        self.run = run

    def train_one_epoch(self, train_loader) -> float:
        self.model.train()
        total_loss = 0.0
        total_samples = 0

        for inputs, targets in train_loader:
            inputs, targets = inputs.to(self.config.device), targets.to(self.config.device)

            self.optimizer.zero_grad()
            outputs = self.model(inputs)
            loss = self.criterion(outputs, targets)
            loss.backward()
            self.optimizer.step()

            batch_size = inputs.size(0)
            total_loss += loss.item() * batch_size
            total_samples += batch_size

        if total_samples == 0:
            return 0.0

        avg_loss = total_loss / total_samples
        return avg_loss

    def validate(self, val_loader) -> Tuple[float, float]:
        self.model.eval()
        running_loss = 0.0
        running_acc = 0.0
        total_samples = 0

        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch = X_batch.to(self.config.device)
                y_batch = y_batch.to(self.config.device)

                logits = self.model(X_batch)
                loss = self.criterion(logits, y_batch)
                batch_acc = accuracy_from_logits(logits, y_batch)

                batch_size = X_batch.size(0)
                running_loss += loss.item() * batch_size
                running_acc += batch_acc * batch_size
                total_samples += batch_size

        if total_samples == 0:
            return 0.0, 0.0

        avg_loss = running_loss / total_samples
        avg_acc = running_acc / total_samples
        return avg_loss, avg_acc

    def fit(self, train_loader, val_loader) -> Dict[str, float]:

        # Sanity check: Verify the batch sizes match the config supplied
        if hasattr(train_loader, 'batch_size') and train_loader.batch_size is not None:
            if train_loader.batch_size != self.config.trainer_batch_size:
                raise ValueError(f"Train loader batch size ({train_loader.batch_size}) does not match config ({self.config.trainer_batch_size})")

        for epoch in range(self.config.num_epochs):
            train_loss = self.train_one_epoch(train_loader)
            val_loss, val_acc = self.validate(val_loader)
            print(
                f"Epoch {epoch}: Train Loss={train_loss:.4f},"
                f"Val Loss={val_loss:.4f}, Val Acc={val_acc * 100:.2f}%"
            )

        return { "num_epochs": epoch+1,
                 "train_loss": train_loss,
                 "val_loss": val_loss,
                 "val_acc": val_acc }
""")


#### Step 2.5 - Test the <code>Trainer</code> class

OK, we have a model. Let's set up a simple optimizer, criterion, and data loader to test the Trainer class. We'll use the FashionMNIST dataset for this example. We used this dataset in the previous homework. It contains thousands of PIL images over 10 different classifications.

> **Fun facts!**<br> What is a PIL image?
>
> PIL stands for "Python Imaging Library" - it's the original name of the popular Python library for opening, manipulating, and saving many types of image files.
> * In PyTorch's data pipelines, images loaded from disk using torchvision or similar libraries typically start as PIL Image objects, which are essentially in-memory representations of images supporting many operations.
> * PyTorch transforms (like `transforms.ToTensor()`) are often designed to work with PIL Images as input, converting them to tensors for model consumption.

In [28]:
# Define our transform for basic PIL images. convert to tensor (values in [0, 1]) and normalize to [-1, 1]
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Download and create the FashionMNIST training and validation datasets
# Note: FashionMNIST files will be downloaded to the "data" directory relative to this notebook's location.
fashion_train_dataset = datasets.FashionMNIST(
    root="data",
    train=True,
    download=True,
    transform=transform,
)
fashion_test_dataset = datasets.FashionMNIST(
    root="data",
    train=False,
    download=True,
    transform=transform,
)

print(fashion_train_dataset.data.shape)

torch.Size([60000, 28, 28])


Excellent. We have our datasets.

>Did you notice where these data are being downloaded in your notebook? It's one of many good reasons why it's best to run your notebooks from the root project directory. It prevents every one of your assignment folders from accumulating their own data directories.

Let's set up our DataLoaders. **We'll use the batch sizes specified in the trainer configuration.**

> **Fun fact!**<br>Wait, can we use different batch sizes for training and testing?<br><br>
> It can *absolutely* make sense to use different batch sizes for training and testing.
> - During training, a *smaller* batch size (like 32 or 64) provides more frequent weight updates, injects noise that can help optimization, and fits in GPU memory even with backprop state.
> - During evaluation/testing, you don't need gradients, so you can use a *much larger* batch size (like 256 or 512) to maximize throughput and speed up evaluation.
> - The only requirement: batch size must not be so large that you run out of memory.
>

In [29]:

fashion_train_loader = DataLoader(
    fashion_train_dataset,
    batch_size=trainer_config.trainer_batch_size,
    shuffle=True)

fashion_test_loader = DataLoader(
    fashion_test_dataset,
    batch_size=trainer_config.evaluator_batch_size,
    shuffle=False)

from my_engine.trainer import Trainer

Let's try it out! We have our model. We have our data. We have our configurations. All that's left is to instantiate a Trainer object, and let it govern training the model:

## CHECKPOINT 1

In [ ]:

from my_engine.trainer import Trainer

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=trainer_config.learning_rate)

# Set the number of epochs to 3 for this simple test
trainer_config.num_epochs = 3

trainer = Trainer(
    model=model,
    optimizer=optimizer,
    criterion=criterion,
    config=trainer_config
)

# Train for a few epochs on FashionMNIST
results = trainer.fit(fashion_train_loader, fashion_test_loader)
print(f"Results: {results}")

Epoch 0: Train Loss=0.5704, Train Acc=0.7938Val Loss=0.4718, Val Acc=82.78%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 1: Train Loss=0.4176, Train Acc=0.8486Val Loss=0.3982, Val Acc=85.82%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 2: Train Loss=0.3769, Train Acc=0.8637Val Loss=0.3754, Val Acc=86.13%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Results: {'num_epochs': 3, 'train_loss': 0.37689786365032196, 'val_loss': 0.375368963599205, 'val_acc': 0.8613}


Verify that the Trainer is working correctly! You should have seen something similar to the following:
```
Epoch 1: Train Loss=0.5758,Val Loss=0.4435, Val Acc=83.67%
Epoch 2: Train Loss=0.4209,Val Loss=0.3985, Val Acc=85.76%
Epoch 3: Train Loss=0.3852,Val Loss=0.3946, Val Acc=86.02%
Results: {'num_epochs': 3, 'train_loss': 0.3851873769124349, 'val_loss': 0.3945958144903183, 'val_acc': 0.8602}# ```

**DO NOT PROCEED UNTIL YOU HAVE OBSERVED SIMILAR OUTPUT!**

Next, verify that the `Trainer` validation works outside of training:


In [ ]:
val_loss, val_acc = trainer.validate(fashion_test_loader)
print(f"Validation Loss: {val_loss:.4f}, Validation Accuracy: {val_acc * 100:.2f}%")

Validation Loss: 0.3754, Validation Accuracy: 86.13%


You should see the same validation results as reported in your last epoch from training.
```
Validation Loss: 0.3946, Validation Accuracy: 86.02%
```

**DO NOT PROCEED UNTIL YOU HAVE OBSERVED SIMILAR OUTPUT!**

In [ ]:
show_todo("Document the Trainer class and its methods")

#### Step 2.6 - Document your `Trainer` class and its methods!

- Provide a header with your identifying info
- Add docstrings to the `Trainer` class and every method.
    - Your docstrings should clearly describe the purpose of the class or method, its arguments, return values, and any important behavior.
    - Use a standard docstring style (e.g., Google style)
- Review your documentation. If you were new to this codebase, would your docstrings help you understand how to use and extend the `Trainer`?
- Add inline comments in your code to ensure you could read and understand your code a year later

In [ ]:
show_todo("Add optimizer and weight_decay support")

#### Step 2.7 - Add optimizer support in TrainerConfig

Did you notice that our TrainConfig has a `weight_decay` field? However, it doesn't have momentum or the ability to store the optimizer. Moreover, it doesn't do anything with any of these!<br>

1. Remember the method we wrote in the previous assignment? Go back to your hw04 notebook. Search for `make_optimizer`. Copy it to your utils.py file. Modify it so that the header looks as follows:
    ```python
    def make_optimizer(params, config: TrainerConfig) -> torch.optim.Optimizer:
        """
        Factory for optimizers.

        Args:
            params (Iterable[torch.nn.Parameter]): Parameters to optimize.
            config (TrainerConfig): Configuration for the optimizer.

        Returns:
            torch.optim.Optimizer: Configured optimizer instance.
        """
        pass
    ```
    <br>
    Finish the function! Modify it to properly handle working with finding its parameters from TrainerConfig.<br><br>

2. Add the two fields `momentum` and `optimizer_name` to the `TrainerConfig` class.

    ```
    optimizer_name: str = "adam"
    momentum: float = 0.9
    ```

That was easy. We won't test it yet... let's continue...


In [ ]:
show_todo("Improve the Trainer class, including adding W&B support")

#### Step 2.8 - Improve Trainer

You'll notice the `Trainer` class has a lot of work ahead to become what we need it to be. Let's keep going.

Perform the following improvements:

1. **Return training accuracy:** Improve `train_one_epoch` to return training loss *and* training accuracy, just like `validate` returns the validation loss and accuracy.

> Did you notice that `run` parameter in the constructor? It's your wandb handle! If you pass a run instance, then you know that wandb has been initialized outside of `Trainer`. You should log everything with this `run` object. The rest of these steps have you complete this.

2. **Add ModelConfig and TrainerConfig Logging:** Modify the `Trainer` class's `fit` method so it updates the self.run.config at the start of fit.
3. **Logs train and validation metrics:** Log metrics computed after each epoch. Use `self.run.log({...})`. This should be done in the `fit` method, not the `train_one_epoch` or `validate` methods.
4. **Add Model Watching:** When a W&B run is used, call `wandb.watch(self.model, ...)` after model initialization (inside the Trainer's `__init__`), so model gradients and parameters are logged to W&B. You might want to do this only if `self.run` is not `None` to avoid unnecessary watching.
5. **Check for Duplicated Watching:** Make sure you only call `wandb.watch` once for a model/run combination.
6. Add a finish() method to Trainer that unwatches the model and finishes the wandb run.

**Be sure not to log anything to wandb if `self.run` is `None`.**



In [ ]:
show_todo("Test out your W&B support!")

Excellent! Let's put it all together and see where we're at!


In [ ]:
# Initialize a new run to our own local W&B space.
run = None
if DO_WANDB_LOGGING:
    run = wandb.init(
        project="csci357-hw05-simple-1",
        name="trainer-test-1",
        reinit=True,
        settings=wandb.Settings(x_stats_sampling_interval=2.0)
    )
if RUN_TRAINING_MODE == RunTrainingMode.FULL:
    num_epochs = 10
else:
    num_epochs = 3

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cb073 (cb073-bucknell-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [ ]:
trainer_config = TrainerConfig(learning_rate=0.001,
                               num_epochs=num_epochs,
                               device=accel_device,
                               optimizer_name="adam",
                               weight_decay=1e-5)
model_config = ModelConfig(model_type="mlp",
                           hidden_units=[512,128],
                           dropout=[0.2,0.1])
model = build_model(num_inputs=784,
                    num_outputs=10,
                    config=model_config)

trainer = Trainer(
    model=model,
    optimizer=make_optimizer(model.parameters(), config=trainer_config),
    criterion=nn.CrossEntropyLoss(),
    config=trainer_config,
    run=run
    )

# Train for a few epochs on FashionMNIST
results = trainer.fit(fashion_train_loader, fashion_test_loader)
trainer.finish_run()

print(f"Results: {results}")

Epoch 0: Train Loss=0.5181, Train Acc=0.8097Val Loss=0.4265, Val Acc=84.53%
Epoch 1: Train Loss=0.3978, Train Acc=0.8540Val Loss=0.4245, Val Acc=84.58%
Epoch 2: Train Loss=0.3620, Train Acc=0.8665Val Loss=0.4192, Val Acc=85.02%
Epoch 3: Train Loss=0.3392, Train Acc=0.8744Val Loss=0.3687, Val Acc=86.71%
Epoch 4: Train Loss=0.3257, Train Acc=0.8801Val Loss=0.3720, Val Acc=86.48%
Epoch 5: Train Loss=0.3072, Train Acc=0.8858Val Loss=0.3840, Val Acc=85.73%
Epoch 6: Train Loss=0.2998, Train Acc=0.8893Val Loss=0.3606, Val Acc=86.79%
Epoch 7: Train Loss=0.2885, Train Acc=0.8916Val Loss=0.3441, Val Acc=87.60%
Epoch 8: Train Loss=0.2796, Train Acc=0.8951Val Loss=0.3294, Val Acc=87.65%
Epoch 9: Train Loss=0.2734, Train Acc=0.8968Val Loss=0.3359, Val Acc=88.15%


epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▅▆▆▇▇▇███
train_loss,█▅▄▃▂▂▂▁▁▁
val_acc,▁▁▂▅▅▃▅▇▇█
val_loss,██▇▄▄▅▃▂▁▁
epoch,9
train_acc,0.8968
train_loss,0.27337
val_acc,0.8815
val_loss,0.33587


Results: {'num_epochs': 10, 'train_loss': 0.2733677619218826, 'val_loss': 0.33586607246398925, 'val_acc': 0.8815}


Be sure to verify your results on W&B. Verify that all your model_config and trainer_config parameters are completed! And **appreciate how simple the above notebook cell is!

In [ ]:
show_warning("DO NOT PROCEED UNLESS YOU HAVE A SOLID UNDERSTANDING OF WHAT YOU JUST DID! AI tools make it too easy to write all your code in one shot. That's dangerous. The latest iterations of these tools, including ChatGPT 5.3 Codex and Claude Code with Opus 4.6, can solve big problems, but the code they are generating is often highly inefficient with some researchers demonstrating complex bugs and issues that are difficult to debug. They are fantastic at small, well defined problems. Use it wisely! Don't fall into that of thinking you can let it solve big problems. You need to be the interactive, engaged designer and architect of your own codebase. You need to be the one who understands the problem and the solution, and the one who can iterate and improve the codebase!")

## Section 3: Engineering Resilience In our Engine

We have successfully shifted from archaic, ridiculous sized "monolithic notebooks" to a
modular "Engine" approach. We emphasized a **separation of responsibilities** in our design.  This is good stuff,
but we're far from finished. We have some work ahead, but by the time we're done with our work this week, we will be well-prepped for the increasingly complex models in the coming weeks. And more importantly, you'll have a solid framework for implementing high-quality models for the rest of the semester and beyond.

**A "Script" runs once and dies. An "Engine" persists.**

Now, let's add resiliance to our engine.

### Checkpointing

Imagine this: You are training a massive Transformer model. It takes 3 days on an A100 GPU (costing ~$4/hr).
You are at hour 71. The loss is looking beautiful. Suddenly...

*   **Your internet connection blips.**
*   **Google Gemini decides it's done with you, disconnecting you from Colab.**
*   **Your cat steps on the power strip.**
*   **Your computer locks up because you ran out of memory or overheated your GPU.**

![Google Colab Sleepy](https://www.eg.bucknell.edu/~brk009/wp-content/uploads/2026/02/GoogleColabSleepy.png)
<br><br>
*Google Colab says, "Ehhh.. I'm bored. Disconnecting..."*

Which position will you be in?<br>
* If you are a **Novice**, you lose everything. You cry. You pay another $300 to restart from Epoch 1. Your boss takes
it out of your paycheck... if you still have a job.
* If you are an **Engineer**, you smile. You load the **checkpoint** from Hour 70. You lose 60 minutes of work, not 3 days. You finish the run.

> Neural net training is **stateful**. Being "stateful" means that the system maintains and depends on its internal state at any given time while some process is executing, and that state evolves over time. It's dynamic! It is not just the model weights that matter! It is the **Optimizer State** (e.g. momentum buffers), the **Learning Rate Scheduler State** (which we'll see soon), and the current **Epoch**.
>
> To build a robust engine, we must be able to **serialize** (save) the state of our training session to disk and **deserialize** (load) it back to resume exactly where we left off.

We're going to be executing much more substantial training sessions in the coming weeks. You don't want the above
scenario to be you! In this section, we will upgrade our `Trainer` class to support **Checkpointing** and **Early Stopping**.


In [ ]:
show_todo("Work through these steps to implement checkpointing")

### Step 3.1 - Adding checkpointing settings to TrainerConfig

Add the following lines to your TrainerConfig definition in `src/my_engine/config.py`:<br>
```python
    # Checkpointing Settings
    checkpoint_dir: str = "./checkpoints"               # Directory where checkpoints will be saved
    checkpoint_last_filename: str = "last.pt"           # Filename for most recent checkpoint
    checkpoint_save_interval: int = 5                   # Save checkpoint every N epochs
    checkpoint_best_filename: str = "best.pt"           # Filename for the best model checkpoint
```

> **Google Colab Users:** The directory should be fine assuming you properly set the project path variables at the top of this folder to be a location off of your mounted Google Drive folder.


### Step 3.2 - Implement `get_architecture_config` method in `MLP_Model` class

This is a helper method. It should do nothing but return a dictionary of every attribute that you created in the model.
<br><br>HINT:
```python
   return {
       'model_type': self.config.model_type,
       'num_inputs': self.num_inputs,
       'num_outputs': self.num_outputs,
       'config': asdict(self.config)
   }
```

### Step 3.3 - Implement `save_checkpoint` in `Trainer`

1. Add new state variables to the constructor:
```python
    # Initialize state variables for checkpointing and early stopping
    self.start_epoch = 0                # Use for resuming training from checkpoint
    self.current_epoch = 0              # Tracks current epoch during training
    self.best_val_loss = float('inf')   # Default best validation loss
    self.patience_counter = 0           # How many epochs without improvement before stopping
```

2. Implement a `save_checkpoint` method in `Trainer`<br><br>

To ensure you have the correct method, we'll give you the entire method:<br>
```python
    def save_checkpoint(self, is_best: bool = False) -> None:
        os.makedirs(self.config.checkpoint_dir, exist_ok=True)
        filepath = os.path.join(self.config.checkpoint_dir, self.config.checkpoint_last_filename)

        checkpoint = {
            # Model weights (the numbers)
            'model_state_dict': self.model.state_dict(),

            # Architecture specification (the blueprint)
            'model_architecture': self.model.get_architecture_config() if hasattr(self.model,
                                                                                    'get_architecture_config') else None,

            # Training state
            'trainer_config': asdict(self.config),
            'optimizer_state_dict': self.optimizer.state_dict(),
            'best_val_loss': self.best_val_loss,
            'epoch': self.current_epoch,
            'patience_counter': self.patience_counter,
        }

        if is_best:
            best_path = os.path.join(self.config.checkpoint_dir, self.config.checkpoint_best_filename)
            torch.save(checkpoint, best_path)
            print(f"--> New best checkpoint saved: {best_path}")
            print(f"--> Also saving as last checkpoint: {filepath}")
        else:
            print(f"--> Saving checkpoint: {filepath}")
        torch.save(checkpoint, filepath)
```

3. Analyze the method. Use AI to understand it. Use AI to document it with a Google-style docstring.


### Step 3.4 - Implement the `load_checkpoint` method in `Trainer`
As you'd expect, the `load_checkpoint` method is pretty much the inverse of the `save_checkpoint` method.
*   Load the file using `torch.load`, doing error checking (paying attention to whether you're loading the last or the best checkpoint)
*   Apply the state dicts to the model and optimizer.
*   **Critical Detail:** You must update `self.start_epoch`. If we load a checkpoint from Epoch 5, the loop in `fit()` must start at Epoch 6, not Epoch 0 (or 1 if you use epoch+1).

The header is as follows:<br>
```python
    def load_checkpoint(self, retrieve_best: bool = False) -> None:
```

### Step 3.5 - Implement checkpointing and Early Stopping in the `fit()` function

1. Modify the loop to use `self.current_epoch` and `self.start_epoch`. Get rid of the simple `epoch` variable.

2. Add a new parameter to `fit` called `resume_from_last_checkpoint`, initialized to False. This serves the purpose of starting the loop by first loading the last checkpoint. How?<br><br>
    At the start of the `fit` method, add:<br>
```python
    # Resume from checkpoint if specified
    if resume_from_last_checkpoint:
        self.load_checkpoint(retrieve_best=False)
```

3. Implement periodic checkpoint saving every N epochs. It'll probably look something like this:<br>
```python
    # Periodic checkpoint saving (every N epochs)
    if (self.current_epoch + 1) % self.config.checkpoint_save_interval == 0:
        self.save_checkpoint(is_best=False)

```

4. Implement saving the best model found so far.

5. Implement early stopping using `self.patience_counter`. Inside the loop, you need to implement logic to track validation loss:
    *   If `val_loss < self.best_val_loss - self.config.early_stopping_min_delta`: Save a checkpoint named `best_model.pt`. Reset your patience counter.
    *   If `val_loss` stops improving for `self.config.early_stopping_patience` epochs: Stop the loop early.



### Step 3.6 - Implement `load_model_from_checkpoint` in `utils.py`


Here's the docstring for the method:
```python
def load_model_from_checkpoint(
    checkpoint_path: Union[str, Path],
    device: torch.device = torch.device('cpu')
) -> nn.Module:
    """Reconstructs any model from a checkpoint file.

    This factory function inspects the checkpoint's model_architecture to
    determine the model type, then dispatches to the appropriate constructor.

    NOTE: This ONLY restores the model architecture and weights, not optimizer state or other metadata.

    Args:
        checkpoint_path: Path to checkpoint file
        device: Device to load onto (default: CPU)

    Returns:
        Reconstructed model with loaded weights

    Raises:
        ValueError: If model_type in checkpoint is unrecognized
        FileNotFoundError: if the checkpoint file does not exist
        KeyError: if the checkpoint is missing the model_architecture metadata
    """
```

This is a pretty simple function because you are ONLY retrieving the model info from the checkpoint specified. You'll check out the  model type and instantiate the model. Right now, we only have a 'mlp' model type, so you'll instantiate a `MLP_Model` with the correct parameters, and then make sure to restore the model weights with the model state_dict, and move it to the correct
device specified in the parameter.


### Step 3.7 - Update `.gitignore` to avoid storing checkpoints
Add the following lines to your `.gitignore` file (which you may already have, but verify! You don't want to be storing
checkpoint files in your git repository. This helps keep your repository clean and prevents accidental commits of
large files.

```
# Data, checkpoints, and other artifacts
data/
checkpoints/
models/         # If they save raw models here
*.pt
*.pth
*.ckpt

# Local Configuration & Secrets
.env
Weights & Biases
wandb/
```

### Step 3.8: The "Crash Test"

Once you have made all your modifications, run the cell below. It simulates a training run that gets interrupted, destroys the trainer object, and then attempts to resume from disk.

**If your implementation is correct, the second training run will start exactly where the first one left off (e.g., Epoch 4 or 6), NOT at Epoch 1.**
<br>
**Use the cell below to verify your implementation.** We will simulate a crash!

In [ ]:
import shutil

# 1. Configure a short run
crash_test_config = TrainerConfig(
    learning_rate=0.001,
    num_epochs=10,
    trainer_batch_size=fashion_train_loader.batch_size, # Make sure we're matching the train loader batch size
    device=accel_device,
    early_stopping_patience=None,  # Disable early stopping
    checkpoint_save_interval=5  # Save every 5 epochs
)

# 2. Let's delete any previous tests
if os.path.exists(crash_test_config.checkpoint_dir):
    shutil.rmtree(crash_test_config.checkpoint_dir)
    print(f"✓ Deleted old checkpoints folder: {crash_test_config.checkpoint_dir}")

# 3. Build a new model and trainer
model = build_model(784, 10, model_config)
optimizer = make_optimizer(model.parameters(), crash_test_config)
criterion = nn.CrossEntropyLoss()
trainer = Trainer(model, optimizer, criterion, crash_test_config)

print("\n--- PHASE 1: Training for 3 epochs then 'Crashing' ---")
trainer.fit(fashion_train_loader, fashion_test_loader, override_num_epochs=5)

print("\n*** CRASH! SYSTEM SHUTDOWN! ***")
del trainer, model, optimizer
import gc
gc.collect()
print("✓ Trainer object deleted. Memory cleared.\n")


--- PHASE 1: Training for 3 epochs then 'Crashing' ---
Epoch 0: Train Loss=0.5648, Train Acc=0.7965Val Loss=0.4368, Val Acc=84.11%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 1: Train Loss=0.4144, Train Acc=0.8507Val Loss=0.4078, Val Acc=84.83%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 2: Train Loss=0.3756, Train Acc=0.8638Val Loss=0.3687, Val Acc=86.43%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 3: Train Loss=0.3546, Train Acc=0.8710Val Loss=0.3694, Val Acc=86.64%
Epoch 4: Train Loss=0.3362, Train Acc=0.8767Val Loss=0.3495, Val Acc=87.54%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
--> Saving checkpoint: ./checkpoints/last.pt

*** CRASH! SYSTEM SHUTDOWN! ***
✓ Trainer object deleted. Memory cleared.


In [ ]:
# --- PHASE 2: The Recovery ---
print("--- PHASE 2: Resuming from Checkpoint ---")

# Re-instantiate fresh model and optimizer with a new trainer using same config
model_recovered = build_model(784, 10, model_config)
optimizer_recovered = make_optimizer(model_recovered.parameters(), crash_test_config)
trainer_recovered = Trainer(model_recovered, optimizer_recovered, criterion, crash_test_config)

try:
    # Attempt recovery
    print("\n[PHASE 2] Resuming training (should continue from epoch 5)...")
    results = trainer_recovered.fit(
        fashion_train_loader,
        fashion_test_loader,
        resume_from_last_checkpoint=True
    )

    print("✅ TEST PASSED: Checkpoint recovery successful!")
    print(f"Final metrics: {results}")

except Exception as e:
    print(f"❌ TEST FAILED: {e}")
    import traceback

    traceback.print_exc()

--- PHASE 2: Resuming from Checkpoint ---

[PHASE 2] Resuming training (should continue from epoch 5)...
Epoch 5: Train Loss=0.3258, Train Acc=0.8795Val Loss=0.3628, Val Acc=87.01%
Epoch 6: Train Loss=0.3134, Train Acc=0.8861Val Loss=0.3506, Val Acc=87.40%
Epoch 7: Train Loss=0.3035, Train Acc=0.8878Val Loss=0.3567, Val Acc=86.94%
Epoch 8: Train Loss=0.2932, Train Acc=0.8921Val Loss=0.3458, Val Acc=87.89%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 9: Train Loss=0.2858, Train Acc=0.8950Val Loss=0.3466, Val Acc=87.99%
--> Saving checkpoint: ./checkpoints/last.pt
✅ TEST PASSED: Checkpoint recovery successful!
Final metrics: {'num_epochs': 10, 'train_loss': 0.2858066107590993, 'val_loss': 0.34663150581121444, 'val_acc': 0.8799}


If you get a TEST PASSED indicator, and Epoch #5 (technically, the 6th epoch) clearly shows you resumed where you left off from epoch 5, then you are good! Congrats!

## Section 4 - Hyperparameter Tuning with W&B Sweeps


### What Are W&B Sweeps and Why Do We Need Them?

Up until now, you've been using Weights & Biases (W&B) for **logging and tracking** individual training runs. When you call `wandb.init()`, you create a single run that logs metrics (loss, accuracy) and hyperparameters. This is great for monitoring one experiment at a time, but what if you want to compare **dozens or hundreds** of different hyperparameter combinations?

That's where **W&B Sweeps** come in.

W&B Logging vs. W&B Sweeps

| Feature | W&B Logging (What You've Done) | W&B Sweeps (What We're Learning) |
|---------|-------------------------------|----------------------------------|
| **Purpose** | Track a single training run | Automate many training runs with different hyperparameters |
| **How it works** | You manually call `wandb.init()`, train, log metrics | W&B controller calls your training function repeatedly with different configs |
| **Hyperparameter selection** | You hardcode values (e.g., `lr=0.001`) | W&B selects values from a search space you define |
| **Result comparison** | Manual (open multiple runs in dashboard) | Automatic (sweep dashboard shows all runs, best configs) |
| **Use case** | "Did this model train well?" | "What's the best learning rate, batch size, architecture?" |

**In short:** W&B logging is for *monitoring* a single experiment. W&B sweeps are for *hyperparameter optimization* across many experiments.

---

### How W&B Sweeps Work

A sweep has three main components:

1. **Sweep Configuration**: A Python dictionary that defines:
   - Which hyperparameters to search (e.g., learning rate, batch size, hidden units)
   - The search method (grid, random, or Bayesian)
   - The metric to optimize (e.g., minimize validation loss)

2. **Training Function**: A Python function that:
   - Reads hyperparameters from `wandb.config` (injected by the sweep controller)
   - Builds a model and trainer with those hyperparameters
   - Trains for a few epochs and logs metrics

3. **Sweep Agent**: W&B's controller that:
   - Calls your training function multiple times
   - Each time, it picks a new set of hyperparameters based on your sweep config
   - Logs all results to the W&B dashboard for easy comparison



### The `method` Parameter in your Sweep Configuration

The most important choice in your sweep configuration is the **search method**. This determines *how* W&B selects hyperparameter combinations to evaluate:

#### 1. **Grid Search** (`method: "grid"`)
- **What it does**: Tries **every possible combination** of hyperparameters you specify.
- **When to use**: When you have a small search space (e.g., 2 batch sizes × 2 learning rates × 2 architectures = 8 runs).
- **Pros**: Exhaustive—guarantees you'll find the best combination in your search space.
- **Cons**: Exponentially expensive. If you have 5 hyperparameters with 3 values each, that's 3^5 = 243 runs!

**Example:**

```python
"method": "grid",
"parameters": {
    "batch_size": {"values": [32, 64]},
    "learning_rate": {"values": [0.001, 0.01]}
}
```

This will run **4 experiments**: (32, 0.001), (32, 0.01), (64, 0.001), (64, 0.01).

#### 2. **Random Search** (`method: "random"`)
- **What it does**: Randomly samples hyperparameter values from the ranges you specify.
- **When to use**: When grid search is too expensive, or you want to explore a large space quickly.
- **Pros**: Often finds good solutions faster than grid search (research shows random search is surprisingly effective).
- **Cons**: No guarantee you'll find the absolute best—might miss the perfect combination.

**Example:**

```python
"method": "random",
"parameters": {
    "batch_size": {"values": [32, 64, 128]},
    "learning_rate": {"min": 0.0001, "max": 0.01}  # Sample from continuous range
}
```

You specify `count=20` when calling `wandb.agent()`, and W&B will run 20 random combinations.

#### 3. **Bayesian Optimization** (`method: "bayes"`)
- **What it does**: Uses past results to *intelligently guess* which hyperparameters to try next.
- **When to use**: When each run is **expensive** (e.g., training a large model for hours), and you want to find good hyperparameters with fewer experiments.
- **Pros**: Most sample-efficient—learns from previous runs to focus on promising regions of the search space.
- **Cons**: More complex; needs at least a few runs to "warm up" before it gets smart.

**How it works (simplified):**
1. Run a few random experiments to gather initial data.
2. Build a probabilistic model: "If learning_rate=0.005 gave good results, maybe 0.004 will too?"
3. Use this model to pick the next hyperparameters to try (balance exploration vs. exploitation).
4. Repeat: train → update model → pick next hyperparameters → train again.

**Example:**

```python
"method": "bayes",
"metric": {"name": "val_loss", "goal": "minimize"},
"parameters": {
    "learning_rate": {"min": 0.0001, "max": 0.01},
    "hidden_units": {"values": [[128, 64], [256, 128], [512, 256]]}
}
```
<br>

#### **Summary: Which Method Should You Use?**

| Method | Best For | Example Use Case |
|--------|----------|-----------------|
| **`grid`** | Small search spaces, want to try everything | "I have 2 batch sizes and 2 learning rates—let's try all 4 combos" |
| **`random`** | Larger search spaces, want to explore quickly | "I want to test 20 random configs from a wide range" |
| **`bayes`** | Expensive runs, want to find good configs with fewest experiments | "Each run takes 2 hours—I can only afford 10 runs total" |

**For this exercise, we'll start with `"grid"`** because you'll test a small, manageable search space. Later, you can experiment with `"random"` and `"bayes"` on your own.


### TODO - Create the `src/my_engine/sweep.py` module

Run the cell below to create the `src/my_engine/sweep.py` module. We'll build on this a bit later. These are two helper functions that will be useful for checking the status of a sweep and terminating a sweep if it did not finish.


In [30]:
if os.path.exists("src/my_engine/sweep.py"):
    print("Warning: src/my_engine/sweep.py already exists. Not overwriting.")
else:
    from IPython import get_ipython
    ipython = get_ipython()
    ipython.run_cell_magic("writefile", "src/my_engine/sweep.py", """
import os
import torch
import wandb

def print_sweep_info(sweep_id: str) -> None:
    api = wandb.Api()
    sweep = api.sweep(sweep_id)
    print(f"Sweep {sweep_id} has {len(sweep.runs)} runs")
    print(f"Sweep {sweep_id} expected {sweep.expected_run_count} runs")
    print(f"Sweep {sweep_id} current state is: {sweep.state}")

def terminate_sweep(sweep_id: str) -> bool:
    api = wandb.Api()
    sweep = api.sweep(sweep_id)
    print(f"Sweep {sweep_id} current state is: {sweep.state}")
    if sweep.state != "FINISHED":
        s = sweep.entity + '/' + sweep.project + '/' + sweep.name
        print(f"Stopping sweep {s}")
        exit_code = os.system('wandb sweep --stop ' + s)
        if exit_code != 0:
            print(f"Failed to stop sweep {s}")
            print(f"Exit code: {exit_code}")
            return False
        else:
            print(f"Sweep {s} stopped successfully")
            return True
    else:
        print(f"Sweep {sweep_id} is already finished")
        return True
""")

show_todo("Verify that you have `src/my_engine/sweep.py`. Document it. Then come back here.")

Writing src/my_engine/sweep.py


In [31]:
show_todo("Create your first W&B sweep")

### Exercise: Your First W&B Sweep

In this exercise, you'll create a simple sweep to find good hyperparameters for Fashion-MNIST. We'll keep it small so it runs quickly:

- **2 batch sizes**: 32, 64
- **2 learning rates**: 0.001, 0.01
- **2 hidden unit configurations**: [128, 64], [256, 128]

This gives us **2 × 2 × 2 = 8 total runs** using grid search.

---

#### Step 4.1 — Define the Sweep Configuration

Run the cell below to define the sweep configuration. Read the comments carefully to understand each part.


In [47]:
# Define the sweep configuration
if RUN_TRAINING_MODE == RunTrainingMode.FULL:
    num_epochs = 10
else:
    num_epochs = 3

my_sweep_config = {
    # Search method: "grid" tries every combination
    "method": "grid",

    # Metric to optimize: We want to minimize validation loss
    # Specify this to help W&B understand what you're trying to optimize
    "metric": {
        "name": "val_loss",
        "goal": "minimize"
    },

    # Hyperparameters to search.
    # Use "value" for fixed parameters, and "values" for a list of values to be swept over.
    "parameters": {
        "trainer_batch_size": {
            "values": [32, 64]
        },

        "learning_rate": {
            "values": [0.001, 0.01]
        },

        "hidden_units": {
            "values": [[128, 64], [256, 128]]
        },

        # Fixed parameters (same for all runs)
        "evaluator_batch_size": { "value": 256 },
        "num_epochs": {"value": num_epochs},  # Keep training short for testing
        "dropout": {"value": [0.2, 0.1]}  # Fixed dropout rates
    }
}

#### Step 4.2 — Initialize the Sweep

Now we'll register this sweep configuration with W&B. This creates a sweep on W&B's servers and returns a unique `sweep_id` that we'll use to run the sweep.


In [48]:
# Initialize the sweep (this creates it on W&B's servers)
sweep_id = wandb.sweep(
    sweep=my_sweep_config,
    project="csci357-hw05-sweeps-intro"
)

Create sweep with ID: vrfsh8lx
Sweep URL: https://wandb.ai/cb073-bucknell-university/csci357-hw05-sweeps-intro/sweeps/vrfsh8lx


In [49]:
# Let's do a sanity check on the sweep...
from my_engine.sweep import print_sweep_info
print_sweep_info(sweep_id)

Sweep vrfsh8lx has 0 runs
Sweep vrfsh8lx expected 8 runs
Sweep vrfsh8lx current state is: PENDING


You should see a new sweep created in the W&B dashboard, and it should be in the "PENDING" state because
you haven't run the sweep agent yet! It will appear in the "RUNNING" state if you had a previous sweep running that is still waiting for runs to complete. And because you are running in "grid" search, it computed the number of runs to expect.

> IF YOU TERMINATE THE SWEEP PREMATURELY IN ANY WAY, THE SWEEP WILL REMAIN OPEN!

Once you run the sweep agent, it will appear in the "RUNNING" state.<br>

Once all the runs are complete, it will appear in the "FINISHED" state.<br>

If you terminate the sweep, it will appear in the "STOPPED" state.<br>

**IMPORTANT**: Periodically groom your dashboard and delete old runs and sweeps. It'll help keep your dashboard clean and organized.

In [35]:
show_todo("Complete the training function for the sweep")

#### Step 4.3 — Write the Training Function

This is the most important part! We need to write a function that:
1. Reads hyperparameters from `wandb.config` (these are the hyperparameters for THIS run, injected by the sweep)
2. Creates a model and trainer with those hyperparameters
3. Trains for a few epochs
4. Logs results to W&B

**Key insight:** The sweep controller will call this function for each hyperparameter combination it has determined to try next. Each time, `wandb.config` will contain different values.

**We'll provide scaffolding to get you started.** Your job is to complete the TODOs.


In [50]:
from my_engine.config import ModelType


def train_sweep():
    # Step 1: Initialize a W&B run
    # The sweep controller will automatically populate wandb.config with hyperparameters
    run = wandb.init(
        project="csci357-hw05-sweeps-intro",
        reinit=True,  # Allow multiple runs in the same notebook
        settings=wandb.Settings(x_stats_sampling_interval=2.0)
    )

    # DONE: Step 2: Read hyperparameters from wandb.config
    # wandb.config is like a dictionary that contains the hyperparameters for THIS run
    config = wandb.config
    print(f"wandb.config: {config}")
    hidden_units = config["hidden_units"]
    trainer_batch_size = config["trainer_batch_size"]
    evaluator_batch_size = config["evaluator_batch_size"]
    learning_rate = config["learning_rate"]
    num_epochs = config["num_epochs"]
    dropout = config["dropout"]

    # Set a meaningful run name based on the hyperparameters
    # This makes it much easier to identify runs in the W&B dashboard
    hidden_str = "x".join(map(str, hidden_units))  # e.g., "128x64"
    run.name = f"bs{trainer_batch_size}_lr{learning_rate}_h{hidden_str}"
    print(f"Run name set to: {run.name}")

    # DONE: Step 3: Create DataLoaders with the batch_size from this run
    # IMPORTANT: We must create NEW dataloaders for each run because batch_size changes!
    train_loader = DataLoader(fashion_train_dataset, batch_size=trainer_batch_size, shuffle=True)

    val_loader = DataLoader(fashion_test_dataset, batch_size=evaluator_batch_size, shuffle=False) # Could set batch size bigger

    # DONE Step 4: Create TrainerConfig and ModelConfig using the hyperparameters from wandb.config
    trainer_config = TrainerConfig(
        trainer_batch_size=trainer_batch_size,
        evaluator_batch_size=evaluator_batch_size,
        learning_rate=learning_rate,
        num_epochs=num_epochs,
    )

    # DONE: Create a ModelConfig with hidden_units and dropout
    model_config = ModelConfig(
        model_type=ModelType.MLP,
        hidden_units=hidden_units,
        dropout=dropout
    )

    # DONE: Step 5: Build the model, optimizer, and criterion with build_model()
    model = build_model(
        num_inputs=784,
        num_outputs=10,
        config=model_config
    )

    # DONE: Use make_optimizer() to create an optimizer using trainer_config
    optimizer = make_optimizer(model.parameters(), trainer_config)

    # Criterion (loss function) is fixed for all runs
    criterion = nn.CrossEntropyLoss()

    # DONE: Step 6: Create the Trainer and train!
    # IMPORTANT: Pass run=run so the Trainer logs to THIS W&B run
    trainer = Trainer(
        model=model, optimizer=optimizer, criterion=criterion, config=trainer_config, run=run
    )

    # Train the model
    results = trainer.fit(train_loader, val_loader)

    # Step 7: Clean up
    trainer.finish_run()

    print(f"✓ Run complete! Final val_loss: {results['val_loss']:.4f}, val_acc: {results['val_acc']*100:.2f}%")

#### Step 4.4 — Run the Sweep Agent

You are going to use your first AI agent to do some work for you. The `wandb.agent()` function will call `train_sweep()` multiple times (once per hyperparameter combination).

**This will take a few minutes** since we're running 8 experiments. Go grab a coffee! ☕


In [37]:
show_todo("Run the sweep agent")


We are going to call [`wandb.agent()`](https://docs.wandb.ai/models/ref/python/functions/agent)
<br><br>
The most critical parameters you pass are:
* `sweep_id`: The ID of the sweep to run. This was returned when you initialized the sweep.
* `project`: [OPTIONAL] The project to run the sweep in. This is the project you initialized the sweep in. As long as you initialized it in your sweep, you don't need to pass it again.
* `entity`: [OPTIONAL] The entity to run the sweep in. Again, as long as you initialized it in your sweep, you don't need to pass it again.
* `function`: The function to call for each hyperparameter combination. This is your `train_sweep()` function.
* `count`: The number of hyperparameter combinations to run. NOTE: If in "grid" search, no need to pass this. It will count the number of combinations to run. WARNING - if you pass a number too large, then the sweep will remain open until you submit more runs!

Once you execute the agent, use the URL that is printed out and watch the results appearing in the W&B dashboard!

In [51]:
# Run the sweep agent
wandb.agent(sweep_id=sweep_id, function=train_sweep)
print("\n" + "="*60)
print("✓ Sweep complete!")
print("\n" + "="*60)

wandb: Agent Starting Run: taqlae4w with config:
wandb: 	dropout: [0.2, 0.1]
wandb: 	evaluator_batch_size: 256
wandb: 	hidden_units: [128, 64]
wandb: 	learning_rate: 0.001
wandb: 	num_epochs: 10
wandb: 	trainer_batch_size: 32
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


wandb.config: {'dropout': [0.2, 0.1], 'evaluator_batch_size': 256, 'hidden_units': [128, 64], 'learning_rate': 0.001, 'num_epochs': 10, 'trainer_batch_size': 32}
Run name set to: bs32_lr0.001_h128x64
Epoch 0: Train Loss=0.5470, Train Acc=0.8005Val Loss=0.4528, Val Acc=83.43%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 1: Train Loss=0.4223, Train Acc=0.8458Val Loss=0.4110, Val Acc=84.81%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 2: Train Loss=0.3890, Train Acc=0.8580Val Loss=0.3852, Val Acc=85.59%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 3: Train Loss=0.3672, Train Acc=0.8663Val Loss=0.3770, Val Acc=86.25%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 4: Train Loss=0.3509, Train Acc=0.8716Val Loss

epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▅▆▆▇▇▇███
train_loss,█▄▃▃▂▂▂▁▁▁
val_acc,▁▄▅▆▇████▇
val_loss,█▅▃▃▂▁▂▂▂▁
epoch,9
train_acc,0.88678
train_loss,0.30537
val_acc,0.8689
val_loss,0.34862


✓ Run complete! Final val_loss: 0.3486, val_acc: 86.89%


wandb: Agent Starting Run: gmrjk9tz with config:
wandb: 	dropout: [0.2, 0.1]
wandb: 	evaluator_batch_size: 256
wandb: 	hidden_units: [128, 64]
wandb: 	learning_rate: 0.001
wandb: 	num_epochs: 10
wandb: 	trainer_batch_size: 64
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


...

wandb.config: {'dropout': [0.2, 0.1], 'evaluator_batch_size': 256, 'hidden_units': [256, 128], 'learning_rate': 0.01, 'num_epochs': 10, 'trainer_batch_size': 64}
Run name set to: bs64_lr0.01_h256x128
Epoch 0: Train Loss=0.6896, Train Acc=0.7568Val Loss=0.5566, Val Acc=80.34%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 1: Train Loss=0.6181, Train Acc=0.7882Val Loss=0.5332, Val Acc=81.13%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 2: Train Loss=0.5960, Train Acc=0.7954Val Loss=0.5483, Val Acc=81.13%
Epoch 3: Train Loss=0.5890, Train Acc=0.7973Val Loss=0.5720, Val Acc=81.41%
Epoch 4: Train Loss=0.5848, Train Acc=0.8001Val Loss=0.4907, Val Acc=82.78%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
--> Saving checkpoint: ./checkpoints/last.pt
Epoch 5: Train Loss=0.5692, Train Acc=0.80

epoch,▁▂▃▃▄▅▆▆▇█
train_acc,▁▅▆▆▇▇▇███
train_loss,█▄▃▃▃▂▂▂▁▁
val_acc,▂▄▄▄▇▆▇▁██
val_loss,▆▅▅▇▂▅▃█▁▂
epoch,9
train_acc,0.81102
train_loss,0.54996
val_acc,0.8332
val_loss,0.49757


✓ Run complete! Final val_loss: 0.4976, val_acc: 83.32%


wandb: Sweep Agent: Waiting for job.
wandb: Sweep Agent: Exiting.



✓ Sweep complete!



Let's check out what the results are...

In [52]:
print_sweep_info(sweep_id)
show_warning("VERIFY THAT ALL RUNS ARE COMPLETED AND STATE IS 'FINISHED'")

Sweep vrfsh8lx has 8 runs
Sweep vrfsh8lx expected 8 runs
Sweep vrfsh8lx current state is: FINISHED


### TERMINATE THE SWEEP IF IT DID NOT FINISH

Sweeps will stay open until you terminate them. You can terminate a sweep right on the W&B dashboard, or you can safely use the following code to execute the shell command for you:

In [53]:
# Plan on using this code to properly terminate your sweep. Only use this if you know you are done with the sweep!
from my_engine.sweep import terminate_sweep
terminate_sweep(sweep_id)

Sweep vrfsh8lx current state is: FINISHED
Sweep vrfsh8lx is already finished


True

#### Step 4.5 — Analyze Results in W&B Dashboard

🎉 **Congratulations!** You've just run your first hyperparameter sweep!

Now open the W&B dashboard (click the link printed above) and explore:

1. **Sweep Overview**: See all 8 runs at a glance. Which run had the lowest `val_loss`?

2. Be sure to check on both the Workspace tab, Runs Tab (which is what you saw before), but most importantly... check out the Sweeps tab!

   - **Parallel Coordinates Plot**: This visualization shows how each hyperparameter affects `val_loss`. Look for patterns:
   - Does a higher learning rate help or hurt?
   - Does batch size matter?
   - Which hidden layer configuration works best?<br>

   - **Parameter Importance**: W&B can calculate which hyperparameters had the biggest impact on your metric. Look for patterns here as well.

## **Section 5: Learning Rate Scheduling**

Learning rate scheduling is one of the most powerful yet underutilized techniques in training deep neural networks. Here's why it matters:

**The Problem with Fixed Learning Rates**<br>

When you use a fixed learning rate throughout training:
- **Early training**: A small learning rate causes slow progress—your model takes forever to find promising regions of the loss landscape
- **Late training**: A large learning rate causes the optimizer to "bounce around" the minimum instead of settling into it, preventing your model from achieving the best possible performance

**The Solution**: Adjust the learning rate during training. Start large to make rapid progress, then gradually decrease it to fine-tune the solution.

### **Why This Matters**

Research has shown that learning rate scheduling can:
- **Improve final accuracy** by 2-5% on complex tasks (that's huge in competitive ML!)
- **Speed up convergence** by allowing aggressive early exploration
- **Enable better generalization** by helping the model settle into wider, more robust minima
- **Recover from plateaus** during training

**Real-world impact**: In the famous ResNet paper (He et al., 2015), the authors used learning rate scheduling to achieve breakthrough performance on ImageNet. Modern SOTA models universally use some form of scheduling.

### Step 5.1 — Understanding Learning Rate Schedulers

PyTorch provides several built-in schedulers in `torch.optim.lr_scheduler`. Here are the most common:

#### `StepLR` — Drop learning rate by a factor every N epochs
```python
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.1)
# Reduces LR by 10x every 10 epochs: 0.01 → 0.001 → 0.0001
```

**When to use**: When you have a rough idea of when training plateaus (e.g., "validation loss stops improving around epoch 20").
<br><br>

#### `ReduceLROnPlateau` — Drop learning rate when a metric stops improving
```python
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=3, verbose=True
)
# After 3 epochs without val_loss improvement, reduce LR by half
```

**When to use**: When you want adaptive scheduling based on actual training progress (most flexible!).




### Step 5.2 — Adding Scheduler Support to TrainerConfig

**TODO**: Add the following fields to your `TrainerConfig` class in `src/my_engine/config.py`:

```python
# Learning Rate Scheduler Settings
use_scheduler: bool = False                                # Enable/disable scheduling
scheduler_type: str = "reduce_on_plateau"                  # Options: "step", "exponential", "cosine", "reduce_on_plateau"
scheduler_step_size: int = 10                              # For StepLR: epochs between LR drops
scheduler_gamma: float = 0.1                               # Factor to reduce LR
scheduler_patience: int = 3                                # For ReduceLROnPlateau: epochs to wait before reducing
scheduler_min_lr: float = 1e-6                             # Minimum learning rate (prevents it from going too low)
```

These fields give you full control over scheduling behavior without cluttering your training code.

### Step 5.3 - Utility function to create a scheduler

Let's create a function that allows us to create a scheduler. Go to your `src/my_engine/utils.py` module. Copy and paste the following function:<br>
```python
def make_lr_scheduler(
    optimizer: torch.optim.Optimizer,
    config: TrainerConfig
) -> Optional[torch.optim.lr_scheduler._LRScheduler]:
    """
    Factory for learning rate schedulers.
    
    Args:
        optimizer: The optimizer to schedule
        config: Configuration containing scheduler settings
        
    Returns:
        Scheduler instance, or None if use_scheduler is False
        
    Raises:
        ValueError: If scheduler_type is unrecognized
    """
    if not config.use_scheduler:
        return None
    
    if config.scheduler_type == "step":
        return torch.optim.lr_scheduler.StepLR(
            optimizer,
            step_size=config.scheduler_step_size,
            gamma=config.scheduler_gamma
        )
    elif config.scheduler_type == "reduce_on_plateau":
        return torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer,
            mode='min',
            factor=config.scheduler_gamma,
            patience=config.scheduler_patience,
            min_lr=config.scheduler_min_lr
        )
    # Add more schedulers as needed...
    else:
        raise ValueError(f"Unknown scheduler type: {config.scheduler_type}")


```

### Step 5.4 — Implementing Scheduler Support in Trainer

**TODO**: Modify your `Trainer` class to support learning rate scheduling. Follow these steps:

#### A. Add scheduler initialization in `__init__`

After initializing the optimizer, add:

```python
# Initialize learning rate scheduler if enabled
self.scheduler = None
if config.use_scheduler:
    self.scheduler = self._create_scheduler()
```


#### B. Modify the `fit()` method to step the scheduler

Inside your training loop (after validation), add:

```python
# Step the learning rate scheduler
if self.scheduler is not None:
    if self.config.scheduler_type == "reduce_on_plateau":
        # ReduceLROnPlateau needs the validation loss
        self.scheduler.step(val_loss)
    else:
        # Other schedulers just need to know an epoch completed
        self.scheduler.step()
    
    # Log current learning rate to W&B
    current_lr = self.optimizer.param_groups[0]['lr']
    if self.run is not None:
        self.run.log({"learning_rate": current_lr}, step=self.current_epoch)
```

**Important**: Notice how `ReduceLROnPlateau` is different! It requires the validation metric to decide when to reduce the learning rate!

#### C. Update checkpointing to save/load scheduler state

In `save_checkpoint()`, add:
```python
'scheduler_state_dict': self.scheduler.state_dict() if self.scheduler else None,
```

In `load_checkpoint()`, add:
```python
if self.scheduler and checkpoint.get('scheduler_state_dict'):
    self.scheduler.load_state_dict(checkpoint['scheduler_state_dict'])
```



### **Step 5.5 — Testing: Fixed LR vs Scheduled LR**

**TODO**: In this cell, you're going to run two experiments on WANDB and compare the results.


In [30]:
# Experiment 1: Fixed learning rate (baseline)
print("=" * 60)
print("EXPERIMENT 1: Fixed Learning Rate")
print("=" * 60)

from my_engine.config import TrainerConfig, ModelConfig
from my_engine.utils import make_optimizer, build_model
from my_engine.trainer import Trainer

run1 = None
if DO_WANDB_LOGGING:
    run1 = wandb.init(
        project="csci357-hw05-lr-scheduling",
        name="fixed_lr",
        reinit=True
    )

if RUN_TRAINING_MODE == RunTrainingMode.FULL:
    num_epochs = 20
else:
    num_epochs = 3

config_fixed = TrainerConfig(
    learning_rate=0.001,
    num_epochs=num_epochs,
    device=accel_device,
    use_scheduler=False,  # No scheduling
    early_stopping_patience=None
)

model_config = ModelConfig(hidden_units=[256, 128], dropout=[0.2, 0.1])
model1 = build_model(784, 10, model_config)
optimizer1 = make_optimizer(model1.parameters(), config_fixed)

# Resetting our loadersjust incase
fashion_train_loader = DataLoader(fashion_train_dataset, batch_size=config_fixed.trainer_batch_size, shuffle=True)
fashion_test_loader = DataLoader(fashion_test_dataset, batch_size=config_fixed.evaluator_batch_size, shuffle=False)

EXPERIMENT 1: Fixed Learning Rate


wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: cb073 (cb073-bucknell-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [31]:
# Instantiate the trainer, fit it, then finish the run.
trainer1 = Trainer(model1, optimizer1, nn.CrossEntropyLoss(), config_fixed, run=run1)

results1 = trainer1.fit(fashion_train_loader, fashion_test_loader)
trainer1.finish_run()

print(f"✓ Fixed LR Final Results: val_loss={results1['val_loss']:.4f}, val_acc={results1['val_acc']*100:.2f}%\n")

Epoch 0: Train Loss=0.5305, Train Acc=0.8071Val Loss=0.4295, Val Acc=84.56%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 1: Train Loss=0.4041, Train Acc=0.8518Val Loss=0.3960, Val Acc=85.73%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 2: Train Loss=0.3676, Train Acc=0.8646Val Loss=0.3799, Val Acc=86.09%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 3: Train Loss=0.3460, Train Acc=0.8716Val Loss=0.3589, Val Acc=87.11%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 4: Train Loss=0.3292, Train Acc=0.8782Val Loss=0.3812, Val Acc=86.40%
--> Saving checkpoint: ./checkpoints/last.pt
Epoch 5: Train Loss=0.3178, Train Acc=0.8830Val Loss=0.3528, Val Acc=87.30%
--> New best checkpoint saved: ./checkpoints/best.pt
--

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▄▅▅▆▆▆▆▇▇▇▇▇▇██████
train_loss,█▅▄▄▃▃▃▃▂▂▂▂▂▂▂▁▁▁▁▁
val_acc,▁▃▃▅▄▅▅▅▆▆▇█▇█▇▇████
val_loss,█▆▅▄▅▃▄▄▂▃▂▂▂▁▂▂▂▂▂▂
epoch,19
train_acc,0.91388
train_loss,0.22657
val_acc,0.8881
val_loss,0.32563


✓ Fixed LR Final Results: val_loss=0.3256, val_acc=88.81%



In [33]:

# Experiment 2: With learning rate scheduling
print("=" * 60)
print("EXPERIMENT 2: Scheduled Learning Rate (ReduceLROnPlateau)")
print("=" * 60)

run2 = None
if DO_WANDB_LOGGING:
    run2 = wandb.init(
        project="csci357-hw05-lr-scheduling",
        name="scheduled_lr",
        reinit=True
    )

config_scheduled = TrainerConfig(
    learning_rate=0.001,  # Same starting LR
    num_epochs=num_epochs,
    device=accel_device,
    use_scheduler=True,
    scheduler_type="reduce_on_plateau",
    scheduler_patience=3,
    scheduler_gamma=0.5,  # Reduce by half when plateaus
    early_stopping_patience=None
)

# DONE: Instantiate the model, optimizer, trainer, fit it, then finish the run
model_config = ModelConfig(hidden_units=[256, 128], dropout=[0.2, 0.1])
model2 = build_model(784, 10, model_config)
optimizer2 = make_optimizer(model2.parameters(), config_scheduled)

# Resetting our loadersjust incase
fashion_train_loader = DataLoader(fashion_train_dataset, batch_size=config_scheduled.trainer_batch_size, shuffle=True)
fashion_test_loader = DataLoader(fashion_test_dataset, batch_size=config_scheduled.evaluator_batch_size, shuffle=False)

trainer2 = Trainer(model2, optimizer2, nn.CrossEntropyLoss(), config_scheduled, run=run2)
results2 = trainer2.fit(fashion_train_loader, fashion_test_loader)
trainer2.finish_run()


# DONE: Print the results. Print a comparison of the val_acc from both runs. Print the percent improvement of the scheduled run over the fixed run.
# NOTE: If you use too few epochs, it's possible that the scheduled run will not outperform the fixed run and may do worse!
print(f"✓ Scheduled LR Final Results: val_loss={results2['val_loss']:.4f}, val_acc={results2['val_acc']*100:.2f}%\n")

# Compare results
print("=" * 60)
print("COMPARISON")
print("=" * 60)
fixed_lr_acc = results1["val_acc"]*100
scheduled_lr_acc = results2["val_acc"]*100
print(f"FIXED LR val_acc: {fixed_lr_acc:.2f} vs. SCHEDULED LR val_acc {scheduled_lr_acc:.2f}")
print(f"Percent Improvement: {scheduled_lr_acc - fixed_lr_acc:.4f}")



EXPERIMENT 2: Scheduled Learning Rate (ReduceLROnPlateau)


Epoch 0: Train Loss=0.5336, Train Acc=0.8053Val Loss=0.4308, Val Acc=84.17%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 1: Train Loss=0.4037, Train Acc=0.8523Val Loss=0.4086, Val Acc=84.73%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 2: Train Loss=0.3668, Train Acc=0.8649Val Loss=0.4039, Val Acc=85.69%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 3: Train Loss=0.3469, Train Acc=0.8727Val Loss=0.3640, Val Acc=86.98%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
Epoch 4: Train Loss=0.3289, Train Acc=0.8794Val Loss=0.3578, Val Acc=87.19%
--> New best checkpoint saved: ./checkpoints/best.pt
--> Also saving as last checkpoint: ./checkpoints/last.pt
--> Saving checkpoint: ./checkpoints/last.pt
Epoch 5: Train Loss=

epoch,▁▁▂▂▂▃▃▄▄▄▅▅▅▆▆▇▇▇██
train_acc,▁▄▄▅▅▆▆▆▆▆▆▆▇▇▇▇████
train_loss,█▅▅▄▄▃▃▃▃▃▃▂▂▂▂▂▁▁▁▁
val_acc,▁▂▃▅▅▄▅▆▆▇▆▇▆▆▆▆████
val_loss,█▇▆▄▄▃▄▂▂▂▂▁▂▂▂▃▁▁▁▁
epoch,19
train_acc,0.9265
train_loss,0.19446
val_acc,0.8972
val_loss,0.31586


✓ Scheduled LR Final Results: val_loss=0.3159, val_acc=89.72%

COMPARISON
FIXED LR val_acc: 88.81 vs. SCHEDULED LR val_acc 89.72
Percent Improvement: 0.9100



**What to look for in W&B**:
1. In the "scheduled_lr" run, look at the `learning_rate` plot—you should see it decrease when validation loss plateaus
2. Compare the validation loss curves—the scheduled run should achieve lower final loss
3. The scheduled run might train slower initially but surpass the fixed LR run later

---


In [ ]:
show_section()

# Challenges

## Challenge 1: Building a factory function that makes `train_sweep` functions

A "factory function" is a function that returns a function. In this case, you'll return a function that will run the training loop for a single sweep run. Sounds complicated, but it's actually quite simple. You already have the beginning of a solid function up above - `train_sweep()`. Now, you need to wrap it in a factory function that will return the training function for a single sweep run.

> Why do this? Because we want to be able to reuse the training function for multiple sweep runs. We don't want to have to rewrite the training sweep function for every single sweep run just because we have different sweep name, entity, project, etc.

You'll also need to add support for the following hyperparameters: optimizer_name, weight_decay, momentum, and early_stopping_patience.

1. Start by copying the function over to a new module file, `src/my_engine/sweep.py`.

2. Use AI to help you encapsulate the train_sweep function inside of a factory function named `make_train_sweep`. It should return the training function for a single sweep run. This function's header should be the following
```python
    def make_train_sweep(wandb_project_name: str,       # string passed to the wandb.init
                        datasets: tuple,                # (train_dataset, val_dataset)
                        device: torch.device,
                        num_inputs: int,
                        num_outputs: int,
                        wandb_entity_name: Optional[str] = None):
```

3. Add support in the encapsulated train_sweep function for the following hyperparameters: optimizer_name, weight_decay, momentum, and early_stopping_patience. Remember that early_stopping_patience can be None to disable early stopping.


In [30]:
show_todo("Test out your train_sweep factory function")

Here's your test. If you did everything correctly, you should be able to execute this cell onto the course team space on W&B:

In [30]:

from my_engine.sweep import make_train_sweep

project = "csci357-hw05-chal01-sweeps"
datasets = (fashion_train_dataset, fashion_test_dataset)
num_inputs = 784
num_outputs = 10

# Test out your train_sweep factory function
train_sweep = make_train_sweep(
    wandb_project_name=project,
    wandb_entity_name=entity,
    datasets=datasets,
    device=accel_device,
    num_inputs=num_inputs,
    num_outputs=num_outputs,
)

In [31]:

num_epochs = 15
if RUN_TRAINING_MODE != RunTrainingMode.FULL:
    num_epochs = 3

chal01_sweep_config = {
    "method": "bayes",
    "metric": {
        "name": "val_loss",
        "goal": "minimize"
    },
    "parameters": {
        # Fixed parameters
        "evaluator_batch_size": { "value": 256 },
        "num_epochs": {"value": num_epochs},
        "early_stopping_patience": { "value": None },

        # Searchable parameters
        "trainer_batch_size": {
            "values": [32, 64]
        },
        # "learning_rate": {
        #     "values": [0.0001, 0.001, 0.01]
        # },
        "learning_rate": {
            "min": 0.00005,
            "max": 0.005,
            "distribution": "log_uniform_values"
        },
        "hidden_units": {
            "values": [[128, 64], [256, 128], [512, 256]]
        },
        "dropout": {
            "values": [[0.0, 0.0], [0.1, 0.2], [0.2, 0.3]]
        },
        "optimizer_name": {
            "values": ["adam", "momentum"]
        },
        "weight_decay": {
            "min": 1e-7,
            "max": 1e-3,
            "distribution": "log_uniform_values"
        },
        "momentum": {
            "min": 0.8,
            "max": 0.99,
            "distribution": "uniform"
        },
    }
}

In [32]:
from my_engine.sweep import print_sweep_info, terminate_sweep

sweep_id = wandb.sweep(
    entity=entity,
    sweep=chal01_sweep_config,
    project=project
)
print_sweep_info(sweep_id)

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc


Create sweep with ID: kuhdrktl
Sweep URL: https://wandb.ai/bucknell-university-csci357-2026sp/csci357-hw05-chal01-sweeps/sweeps/kuhdrktl
Sweep kuhdrktl has 0 runs
Sweep kuhdrktl expected None runs
Sweep kuhdrktl current state is: PENDING


In [33]:
count = 15
if RUN_TRAINING_MODE != RunTrainingMode.FULL:
    count = 5

wandb.agent(sweep_id=sweep_id, function=train_sweep, count=count)


wandb: Agent Starting Run: r98flbqz with config:
wandb: 	dropout: [0.2, 0.3]
wandb: 	early_stopping_patience: None
wandb: 	evaluator_batch_size: 256
wandb: 	hidden_units: [512, 256]
wandb: 	learning_rate: 0.0001383490289750882
wandb: 	momentum: 0.8429425895632165
wandb: 	num_epochs: 15
wandb: 	optimizer_name: momentum
wandb: 	trainer_batch_size: 64
wandb: 	weight_decay: 1.5087395495438827e-05
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: cb073 (bucknell-university-csci357-2026sp) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


wandb.config: {'dropout': [0.2, 0.3], 'early_stopping_patience': None, 'evaluator_batch_size': 256, 'hidden_units': [512, 256], 'learning_rate': 0.0001383490289750882, 'momentum': 0.8429425895632165, 'num_epochs': 15, 'optimizer_name': 'momentum', 'trainer_batch_size': 64, 'weight_decay': 1.5087395495438827e-05}
Run name set to: bs64_lr0.0001383490289750882_h512x256
Could not load checkpoint (Error(s) in loading state_dict for MLP_Model:
	Missing key(s) in state_dict: "layers.3.weight", "layers.3.bias", "layers.6.weight", "layers.6.bias". 
	Unexpected key(s) in state_dict: "layers.2.weight", "layers.2.bias", "layers.4.weight", "layers.4.bias". ), 
STARTING FROM SCRATCH.
Epoch 0:
Train Loss=1.9108
Val Loss=1.4299
Train Acc=0.5341
Val Acc=73.56%
Train F1 Macro=0.5204
Val F1 Macro=0.6983
--> New best checkpoint saved: ./checkpoints/csci357-hw05-chal01-sweeps-best.pt
--> Also saving as last checkpoint: ./checkpoints/csci357-hw05-chal01-sweeps-last.pt


wandb: Ctrl + C detected. Stopping sweep.


In [34]:
print("\n" + "="*60)
print("✓ Sweep complete! All runs finished.")
print("\n" + "="*60)

print_sweep_info(sweep_id)
# Should only be called if we want to NOT resume with checkpointing later
terminate_sweep(sweep_id)
print_sweep_info(sweep_id)


✓ Sweep complete! All runs finished.

Sweep kuhdrktl has 1 runs
Sweep kuhdrktl expected None runs
Sweep kuhdrktl current state is: RUNNING
Stopping sweep bucknell-university-csci357-2026sp/csci357-hw05-chal01-sweeps/kuhdrktl
Sweep bucknell-university-csci357-2026sp/csci357-hw05-chal01-sweeps/kuhdrktl stopped successfully
Sweep kuhdrktl current state is: RUNNING


True

Epoch 1:
Train Loss=1.1203
Val Loss=0.8343
Train Acc=0.7367
Val Acc=77.52%
Train F1 Macro=0.7111
Val F1 Macro=0.7539
--> New best checkpoint saved: ./checkpoints/csci357-hw05-chal01-sweeps-best.pt
--> Also saving as last checkpoint: ./checkpoints/csci357-hw05-chal01-sweeps-last.pt
Epoch 2:
Train Loss=0.7725
Val Loss=0.6525
Train Acc=0.7708
Val Acc=79.19%
Train F1 Macro=0.7603
Val F1 Macro=0.7829
--> New best checkpoint saved: ./checkpoints/csci357-hw05-chal01-sweeps-best.pt
--> Also saving as last checkpoint: ./checkpoints/csci357-hw05-chal01-sweeps-last.pt



3. **Try random search**: Change `method` to `"random"`, add more hyperparameter options, and run 15 random samples. Compare the best result to your grid search.

4. **Try Bayesian optimization**: Change `method` to `"bayes"`, increase `num_epochs` to 15 and run 15 experiments. Does it find better hyperparameters faster than random search?

5. Learn more about sampling from distributions in W&B: https://docs.wandb.ai/models/sweeps/sweep-config-keys#parameters
because "uniform" is the default for any parameter where you specify a range. THat may not always be ideal. In particular, you may want to sample from a log normal space for learning_rate and weight_decay. Make the change to the sweep config to use "log_uniform_values" for learning_rate and weight_decay (which means you'll need to change learning rate to use a min and max value.)

6. **Sweep over architectures**: Try deeper networks (e.g., `[512, 256, 128]`) or different dropout patterns.

7. **Sweep over LR scheduling**: Try different LR scheduling strategies.

Remember: The training function is **reusable**. You can change the `sweep_config` and call `wandb.sweep()` + `wandb.agent()` again without rewriting the training code!





## Challenge 2: Incorporate `torchmetrics`

So far, our Trainer reports **loss** and **accuracy**. Accuracy is a great first metric, but it can be dangerously misleading when:
- Classes are **imbalanced** (e.g., 95% class A, 5% class B)
- Some classes are “easy” and others are “hard”
- The cost of false positives vs false negatives is not symmetric (medical screening, fraud, etc.)

In these settings, accuracy can look “good” while the model completely fails on the classes you actually care about.

### Why F1?
For classification, two key ideas are:
- **Precision**: Of the examples we *predicted* as class $k$, how many were *actually* class $k$? -> Precision = TP / (TP + FP)
- **Recall**: Of the examples that are *actually* class $k$, how many did we correctly find? -> Recall = TP / (TP + FN)
<br/>

Where:
- TP (True Positives): Correctly predicted positive 
- FP (False Positives): Incorrectly predicted positive
- FN (False Negatives): Incorrectly predicted negative
- TN (True Negatives): Correctly predicted negative

The **F1 score** is the harmonic mean of precision and recall.

You can find plenty of info online about these metrics.

For **multiclass** problems, we’ll use **macro-F1**:
- Compute F1 **separately per class**
- Average across classes equally (each class gets the same weight)

Macro-F1 is especially useful when you want to prevent the model from “ignoring” rare classes.

This is a large, open-ended challenge.
1. Verify that `torchmetrics` is in your environment. If not, check out `pyproject.toml` and add it! Import it. Find the API documentation and be ready to use it!
2. Find a dataset on UCI's ML repo or Kaggle that is a classification dataset that is HIGHLY IMBALANCED! Read in the data into a numpy array or pandas dataframe. Separate it into an X for the features and the corresponding y variable representing the target.
3. Perform any normalization/standardization of the attributes if necessary. Remember that for classification, our target has been one variable with different labels for each class, not separate variables for each class.
4. Use stratified random sampling to separate your data into an 80/20 train/test split.
5. Create your tensor train and test datasets.
6. Determine how you want to modularize the performance metrics in the `fit()` function in `Trainer`. This is going to be very useful down the road. Right now, it's hardcoding train_loss, train_acc, val_loss, val_acc. Our aim is to allow us to specify the target metric(s) to compute, and have them returned as a dictionary by fit, AND have them logged in wandb.
7. You should always return the existing metrics. Add metrics for `train_f1_macro` and `val_f1_macro`.

Once you get that working, verified by a single fit run with a Trainer, it's time to set up a sweep to find optimal results. Remember, you should be maximizing val_f1_macro now!

* use our course `entity`
* use `project = "csci357-hw05-chal02-f1"

Have fun!

**Suggestion:**<br>
Not sure where to look for a challenging dataset? Consider this one...

```python
# 1. Download and load the dataset using ucimlrepo
from ucimlrepo import fetch_ucirepo

# 2. Fetch dataset (ID 572 for Taiwanese Bankruptcy Prediction)
taiwanese_bankruptcy = fetch_ucirepo(id=572)

# 3. Set up our X and y data (as numpy arrays or pandas dataframes). Be mindful of types!
X = taiwanese_bankruptcy.data.features.values.astype(np.float32)
y = taiwanese_bankruptcy.data.targets.values.ravel().astype(np.int64)

# 4. Train/test split (e.g. 80/20, stratified on target)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
```

In [30]:
if IN_COLAB:
    !uv pip install -q ucimlrepo
    pass

In [31]:
# ANSWER:
# Create all of the code cells you need to answer this challenge. Clearly document your work.
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset
from sklearn.preprocessing import StandardScaler

# 1. Download and load the dataset using ucimlrepo
from ucimlrepo import fetch_ucirepo

# 2. Fetch dataset (ID 572 for Taiwanese Bankruptcy Prediction)
taiwanese_bankruptcy = fetch_ucirepo(id=572)

# 3. Set up our X and y data (as numpy arrays or pandas dataframes). Be mindful of types!
X = taiwanese_bankruptcy.data.features.values.astype(np.float32)
y = taiwanese_bankruptcy.data.targets.values.ravel().astype(np.int64)

# 4. Train/test split (e.g. 80/20, stratified on target)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

# Set num_inputs and num_outputs
num_inputs = X_train.shape[1]
num_outputs = len(np.unique(y_train))

# Show num_inputs and num_outputs
print(f"Number of input features: {num_inputs}")
print(f"Number of target classes: {num_outputs}")

# Standardize data
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.long) # don't need squeeze since ravel already shrinks the dimension down
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)
print(X_train_tensor.shape)
print(y_train_tensor.shape)
print(y_train[:5])

# Create TensorDataset
tw_train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
tw_test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

Number of input features: 95
Number of target classes: 2
torch.Size([5455, 95])
torch.Size([5455])
[0 0 0 0 0]


In [38]:
from my_engine.config import ModelType

chal02_project = "csci357-hw05-chal02-single"
hidden_units = [128, 64]
learning_rate = 0.001
trainer_batch_size = 32

# Set up wandb run
run = wandb.init(
        project=chal02_project,
        entity=entity,
        reinit=True,
        settings=wandb.Settings(x_stats_sampling_interval=2.0),
    )

# Descriptive run name for the W&B dashboard
hidden_str = "x".join(map(str, hidden_units))
run.name = f"bs{trainer_batch_size}_lr{learning_rate}_h{hidden_str}"
print(f"Run name set to: {run.name}")

# Set up model for single run
model_config = ModelConfig(
    ModelType.MLP,
    hidden_units=hidden_units,
    dropout=[0.1, 0.2],
)
model = build_model(num_inputs, num_outputs, model_config)

# Set up trainer for single run
trainer_config = TrainerConfig(
    trainer_batch_size=trainer_batch_size,
    evaluator_batch_size=128,
    learning_rate=learning_rate,
    device=accel_device,
    optimizer_name="adam",
    checkpoint_last_filename="05-tw-single-last.pt",
    checkpoint_best_filename="05-tw-single-best.pt",
    num_classes=num_outputs,
)
criterion = nn.CrossEntropyLoss()
optimizer = make_optimizer(model.parameters(), trainer_config)
trainer = Trainer(model, optimizer, criterion, trainer_config, run=run)

# Create dataloaders
tw_train_dataloader = DataLoader(tw_train_dataset, trainer_config.trainer_batch_size, shuffle=True)
tw_test_dataloader = DataLoader(tw_test_dataset, trainer_config.evaluator_batch_size, shuffle=False)

# Run the trainer
results = trainer.fit(tw_train_dataloader, tw_test_dataloader, resume_from_last_checkpoint=False)

# Clean up
trainer.finish_run()
print(f"Run complete! Final val_loss: {results['val_loss']:.4f}, val_acc: {results['val_acc']*100:.2f}%")

Run name set to: bs32_lr0.001_h128x64
Epoch 0:
Train Loss=0.1583
Val Loss=0.1555
Train Acc=0.9635
Val Acc=96.99%
Train F1 Macro=0.5006
Val F1 Macro=0.5740
--> New best checkpoint saved: ./checkpoints/05-tw-single-best.pt
--> Also saving as last checkpoint: ./checkpoints/05-tw-single-last.pt
Epoch 1:
Train Loss=0.0901
Val Loss=0.1337
Train Acc=0.9694
Val Acc=96.77%
Train F1 Macro=0.5965
Val F1 Macro=0.5687
--> New best checkpoint saved: ./checkpoints/05-tw-single-best.pt
--> Also saving as last checkpoint: ./checkpoints/05-tw-single-last.pt
Epoch 2:
Train Loss=0.0819
Val Loss=0.1622
Train Acc=0.9705
Val Acc=96.77%
Train F1 Macro=0.6282
Val F1 Macro=0.5687
Epoch 3:
Train Loss=0.0747
Val Loss=0.1558
Train Acc=0.9721
Val Acc=96.48%
Train F1 Macro=0.6681
Val F1 Macro=0.5772
Epoch 4:
Train Loss=0.0707
Val Loss=0.1687
Train Acc=0.9716
Val Acc=96.70%
Train F1 Macro=0.6712
Val F1 Macro=0.5671
--> Saving checkpoint: ./checkpoints/05-tw-single-last.pt
Epoch 5:
Train Loss=0.0666
Val Loss=0.1946
Tr

epoch,▁▂▃▅▆▇█
train_acc,▁▅▅▆▆██
train_f1_macro,▁▄▅▆▆▇█
train_loss,█▃▂▂▁▁▁
val_acc,█▆▆▃▅▄▁
val_f1_macro,▃▁▁▃▁▇█
val_loss,▄▁▄▄▅█▇
epoch,6
train_acc,0.97489
train_f1_macro,0.74264
train_loss,0.06596


Run complete! Final val_loss: 0.1845, val_acc: 96.26%


In [41]:
from my_engine.sweep import make_train_sweep
from my_engine.sweep import print_sweep_info, terminate_sweep

project = "csci357-hw05-chal02-f1"
datasets = (tw_train_dataset, tw_test_dataset)

# Test out your train_sweep factory function
train_sweep = make_train_sweep(
    wandb_project_name=project,
    wandb_entity_name=entity,
    datasets=datasets,
    device=accel_device,
    num_inputs=num_inputs,
    num_outputs=num_outputs,
)


num_epochs = 15
if RUN_TRAINING_MODE != RunTrainingMode.FULL:
    num_epochs = 3

chal02_sweep_config = {
    "method": "bayes",
    "metric": {
        "name": "val_f1_macro",
        "goal": "maximize"
    },
    "parameters": {
        # Fixed parameters
        "evaluator_batch_size": { "value": 128 },
        "num_epochs": {"value": num_epochs},
        "early_stopping_patience": { "value": None },

        # Searchable parameters
        "trainer_batch_size": {
            "values": [16, 32]
        },
        # "learning_rate": {
        #     "values": [0.0001, 0.001, 0.01]
        # },
        "learning_rate": {
            "min": 0.00005,
            "max": 0.005,
            "distribution": "log_uniform_values"
        },
        "hidden_units": {
            "values": [[128, 64], [256, 128], [512, 256]]
        },
        "dropout": {
            "values": [[0.0, 0.0], [0.1, 0.2], [0.2, 0.3]]
        },
        "optimizer_name": {
            "values": ["adam", "momentum"]
        },
        "weight_decay": {
            "min": 1e-7,
            "max": 1e-3,
            "distribution": "log_uniform_values"
        },
        "momentum": {
            "min": 0.8,
            "max": 0.99,
            "distribution": "uniform"
        },
    }
}

sweep_id = wandb.sweep(
    entity=entity,
    sweep=chal02_sweep_config,
    project=project
)
print_sweep_info(sweep_id)

count = 15
if RUN_TRAINING_MODE != RunTrainingMode.FULL:
    count = 5

wandb.agent(sweep_id=sweep_id, function=train_sweep, count=count)

print("\n" + "="*60)
print("✓ Sweep complete! All runs finished.")
print("\n" + "="*60)

# Should only be called if we want to NOT resume with checkpointing later
terminate_sweep(sweep_id)
print_sweep_info(sweep_id)

Create sweep with ID: umths0ec
Sweep URL: https://wandb.ai/bucknell-university-csci357-2026sp/csci357-hw05-chal02-f1/sweeps/umths0ec
Sweep umths0ec has 0 runs
Sweep umths0ec expected None runs
Sweep umths0ec current state is: PENDING


wandb: Agent Starting Run: x6gyersq with config:
wandb: 	dropout: [0, 0]
wandb: 	early_stopping_patience: None
wandb: 	evaluator_batch_size: 128
wandb: 	hidden_units: [512, 256]
wandb: 	learning_rate: 0.00017312914184491646
wandb: 	momentum: 0.8142918849860402
wandb: 	num_epochs: 15
wandb: 	optimizer_name: adam
wandb: 	trainer_batch_size: 16
wandb: 	weight_decay: 4.3693710397500323e-07
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


wandb.config: {'dropout': [0, 0], 'early_stopping_patience': None, 'evaluator_batch_size': 128, 'hidden_units': [512, 256], 'learning_rate': 0.00017312914184491646, 'momentum': 0.8142918849860402, 'num_epochs': 15, 'optimizer_name': 'adam', 'trainer_batch_size': 16, 'weight_decay': 4.3693710397500323e-07}
Run name set to: bs16_lr0.00017312914184491646_h512x256
Epoch 0:
Train Loss=0.1727
Val Loss=0.2236
Train Acc=0.9672
Val Acc=96.70%
Train F1 Macro=0.5026
Val F1 Macro=0.5968
--> New best checkpoint saved: ./checkpoints/csci357-hw05-chal02-f1-best.pt
--> Also saving as last checkpoint: ./checkpoints/csci357-hw05-chal02-f1-last.pt
Epoch 1:
Train Loss=0.1004
Val Loss=0.1796
Train Acc=0.9701
Val Acc=96.63%
Train F1 Macro=0.6365
Val F1 Macro=0.5807
--> New best checkpoint saved: ./checkpoints/csci357-hw05-chal02-f1-best.pt
--> Also saving as last checkpoint: ./checkpoints/csci357-hw05-chal02-f1-last.pt
Epoch 2:
Train Loss=0.0835
Val Loss=0.1617
Train Acc=0.9709
Val Acc=96.63%
Train F1 Macro

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_acc,▁▂▂▂▃▃▄▄▅▅▆▆▇▇█
train_f1_macro,▁▃▄▄▅▅▅▅▆▆▇▇▇██
train_loss,█▄▄▃▃▃▃▃▂▂▂▂▁▁▁
val_acc,█▇▇▄▅▅▆▄▇▇▃█▆▅▁
val_f1_macro,▃▂▂▁▁▁▁▅▅▄█▇▄▇▇
val_loss,▇▃▁▁▂▁▃▃▂▃▂▃▇▇█
epoch,14
train_acc,0.99065
train_f1_macro,0.91913
train_loss,0.02875


Run complete! Final val_loss: 0.2378, val_acc: 96.11%


...

Run complete! Final val_loss: 0.6166, val_acc: 96.11%


wandb: Sweep Agent: Waiting for job.
wandb: Job received.
wandb: Agent Starting Run: 01m8101r with config:
wandb: 	dropout: [0.1, 0.2]
wandb: 	early_stopping_patience: None
wandb: 	evaluator_batch_size: 128
wandb: 	hidden_units: [512, 256]
wandb: 	learning_rate: 0.001332803532661608
wandb: 	momentum: 0.8700987316254077
wandb: 	num_epochs: 15
wandb: 	optimizer_name: adam
wandb: 	trainer_batch_size: 32
wandb: 	weight_decay: 1.2083654971687616e-07
wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.


wandb.config: {'dropout': [0.1, 0.2], 'early_stopping_patience': None, 'evaluator_batch_size': 128, 'hidden_units': [512, 256], 'learning_rate': 0.001332803532661608, 'momentum': 0.8700987316254077, 'num_epochs': 15, 'optimizer_name': 'adam', 'trainer_batch_size': 32, 'weight_decay': 1.2083654971687616e-07}
Run name set to: bs32_lr0.001332803532661608_h512x256
Epoch 0:
Train Loss=0.1332
Val Loss=0.1442
Train Acc=0.9652
Val Acc=96.77%
Train F1 Macro=0.4911
Val F1 Macro=0.4918
--> New best checkpoint saved: ./checkpoints/csci357-hw05-chal02-f1-best.pt
--> Also saving as last checkpoint: ./checkpoints/csci357-hw05-chal02-f1-last.pt
Epoch 1:
Train Loss=0.0894
Val Loss=0.1680
Train Acc=0.9676
Val Acc=96.99%
Train F1 Macro=0.5425
Val F1 Macro=0.5562
Epoch 2:
Train Loss=0.0768
Val Loss=0.1803
Train Acc=0.9701
Val Acc=96.92%
Train F1 Macro=0.6202
Val F1 Macro=0.5883
Epoch 3:
Train Loss=0.0706
Val Loss=0.1791
Train Acc=0.9723
Val Acc=96.85%
Train F1 Macro=0.6921
Val F1 Macro=0.5863
Epoch 4:
Tra

epoch,▁▁▂▃▃▃▄▅▅▅▆▇▇▇█
train_acc,▁▂▂▃▃▄▄▄▅▆▆▇▇▇█
train_f1_macro,▁▂▃▄▅▅▅▆▆▇▇████
train_loss,█▅▄▄▄▃▃▃▂▂▂▁▁▁▁
val_acc,▆█▇▇▄▄▆▅▄▂▅▂▃▄▁
val_f1_macro,▁▃▄▄▄▄▅▃▅▆▄▆▆▇█
val_loss,▁▁▂▂▂▃▂▄▄▅▄▆▄█▇
epoch,14
train_acc,0.98955
train_f1_macro,0.91323
train_loss,0.02735


Run complete! Final val_loss: 0.5772, val_acc: 96.04%

✓ Sweep complete! All runs finished.

Stopping sweep bucknell-university-csci357-2026sp/csci357-hw05-chal02-f1/umths0ec
Sweep bucknell-university-csci357-2026sp/csci357-hw05-chal02-f1/umths0ec stopped successfully
Sweep umths0ec has 15 runs
Sweep umths0ec expected None runs
Sweep umths0ec current state is: FINISHED


# Career Survey

Read the two articles:
* https://www.theguardian.com/us-news/2026/feb/08/ai-washing-job-losses-artificial-intelligence - The Guardian, Feb 8, 2026
* https://www.lowyinstitute.org/the-interpreter/ai-didn-t-fire-you-board-did - The Interpreter, Feb 4, 2026

This begs real answers to the question: Is the AI job apocalypse real? Or is it just a convenient cover? While tech giants like Amazon, HP, and Salesforce announce massive layoffs citing "AI efficiency," economists are calling bullshit. The article suggests that companies may be "AI-washing" their cuts, using your field as a scapegoat while the real culprits -- pandemic overhiring, tariff impacts, and corporate greed -- go unmentioned. It discusses the story of one laid-off Amazon employee was a heavy AI user who built automation tools for her team, yet she was still cut—only to be replaced by a cheaper junior worker, not a chatbot.

As you navigate a tough job market, understanding whether you're actually competing against algorithms or being caught in corporate cost-cutting theater isn't just academic curiosity—it's career survival intelligence that can help you ask the right questions when considering where to apply. Read this Guardian investigation to decode what's really happening to tech jobs, and write a short (2-3 paragraphs) take on this. Are you building the tools that will replace you, or are companies just using AI as convenient cover for decisions they'd make anyway?


**ANSWER:**
<br>
I believe that AI is being used as a cover to lay off workers that companies want to lay off. We can see this in the Guardian article. I personally think that AI is a powerful tool, but it is only as good as its wielder. I think that this notion is not understood by everyone as a lot of hype and big headlines are skewing people's perspectives into believing AI is the end of labor itself. Some companies are taking advantage of this fabrication of reality to lay people off.

However, that is not to say that all companies are like this. I think that IBM recently announced that they were hiring a lot more new grads (after laying off a bunch of people a year ago), which could indicate that they genuinely thought AI could replace some of their workforce. Though, this is a rare case. I think that we will see a gradual increase in hiring of engineers at some of these firms as they make up for the layoffs to make up for the loss of actual talent. This will gradual because these corporations may have learned their lessons of overhiring during the pandemic.
<br>




In [ ]:
display(HTML(css_styles + """
<div class="section-box"><h1></h1></div>
"""))

# Society

Read the article:
https://www.interviewquery.com/p/ai-workload-creep-tech-workers-study - InterviewQuery, Feb 11, 2026

A UC Berkeley field study of knowledge workers using AI finds that AI often intensifies work instead of reducing it, leading to “workload creep.

AI makes individual tasks faster and lowers friction, which causes task expansion (starting more projects), expectation inflation (managers raising output expectations), and increased multitasking that heightens context-switching and pressure. Tech workers feel this early because AI is deeply embedded in coding, documentation, analysis, and debugging, where faster drafting leads to more iterations and more assigned features or dashboards. Time savings are frequently reallocated to verifying and fixing AI output, managing more simultaneous workstreams, and handling extra coordination, which can contribute to burnout as expectations rise faster than capacity. The study argues that AI functions as a work multiplier: without deliberate organizational guardrails—like scope boundaries, protected focus time, and explicit decisions about where productivity gains go—efficiency gains are simply converted into more work, risking workload creep becoming the norm.

* The study suggests AI increases ‘workload creep’ rather than reducing work. In your own experience (courses, internships, research), have AI tools actually reduced your total workload, or just changed its nature? Give a concrete examples.
* If AI lets some students or workers produce more, should professors/managers raise expectations for everyone? How might this affect equity between people with different AI skills or access?
* Metrics and measurement - Design a small empirical study to measure whether AI tools cause workload creep in a software engineering course (or capstone project). What would you measure, and how would you distinguish ‘more output’ from ‘more pressure’?

Write a few paragraphs on this that answers the questions above.

**ANSWER:**
<br>
From my own experience, I believe that AI tools did not reduce my total workload. Instead, it has fundamentally changed the way I do work. Before, I would have to waste time looking for information online, but now, I find myself diving a little deeper into why the AI tool response was like that. Often, I find myself investigating these concepts online (outside of AI tools); AI helps me know where to look to verify that information. For instance, I was looking up information about MySQL metrics that were being stored in the system schema. Llama helped direct me towards the MySQL database and documentation websites, which helped me find the original sources of information.

No, I believe that raising expectations would be a bad policy, and the reason is that AI skill itself isn't evenly distributed. Access varies, prompting fluency varies, and most importantly, judgement to know when AI output is wrong varies directly with domain expertise. That is part of the reason why I try not to use AI to cheat my way out of learning but rather enhance my learning with it.

I would run a study in a software engineering class where students are only allowed to work on a project while being supervised. One group will have access to AI tools and the other without. I would need to measure both objective and subjective signals. On the objective side: track features shipped, commit frequency, lines of code, test coverage, and self-reported hours per week. On the subjective side: run bi-weekly suveys measuring perceived workload, stress levels, and confidence in the correctness of their own code. The critical variable to isolate is whether the AI group produces more without reporting more pressure versuese produces more and reports higher pressure (more workload creep). I would also want to capture whether course staff or project stakeholders raised their informal expectations mid-semester after seeing early AI-assisted output.
<br>




In [ ]:
display(HTML(css_styles + """
<div class="section-box"><h1></h1></div>
"""))

# AI Usage Disclosure

**IDE or browser-based environment used:** VSCode

**AI tools used:** Claude

**Short reflection:**
How did you use AI this week? Did it help you understand activation functions? Debug your training loop? Explain backpropagation? Be specific about what worked well and what didn't.

I used it to understand the google drive connection and colab environment importing.

**AI Chat Dialogs:**
(Paste relevant snippets here - particularly any that helped you understand conceptual questions or debug code)
<br/>
1.
```
why does restarting the kernel properly import/register the package?

4:40 PM
When you run uv pip install with a pyproject.toml file, it installs your package by:

Creating entry in site-packages: It adds metadata and references to Python's site-packages directory that tell Python where to find your package
Modifying sys.path: It updates the paths Python searches when you import something
However, the currently running Python kernel already has its sys.path loaded into memory at startup. When the installation happens mid-session, these changes don't automatically propagate to the running kernel.

Why Restart Fixes It
When you restart the kernel:

Python re-initializes and re-reads all the site-packages metadata
sys.path gets rebuilt from scratch, now including the newly installed package location
Your import my_engine statement can now find the package properly
Why Colab Specifically Has This Issue
In a local environment, you'd typically:

Install packages before starting your Python session, OR
Use tools that handle dynamic reloading better
In Colab:

You're installing packages in the same notebook session you're working in
The kernel is already running when you do the installation
Colab's package registration doesn't trigger the same path updates that would happen in a fresh Python session
It's essentially a state synchronization problem - the running kernel's state is stale compared to what's actually installed on the system.
```
2.
```
❯ can you write docstrings and concise comments for the following methods

⏺ Let me read the file first.

⏺ Read 1 file (ctrl+o to expand)    

⏺ Update(src/my_engine/trainer.py)                                                                                                                                             
  ⎿  Added 11 lines                                                                                                                                                            
      10  from my_engine.utils import accuracy_from_logits                                                                                                                     
      11          
      12  class Trainer:                                                                                                                                                       
      13 +    """A training loop wrapper that handles training, validation, and logging for PyTorch models."""                                                          
      14 +                                                                                                                                                                     
      15      def __init__(                                                                                                                                                  
      16          self,                                                                                                                                                        
      17          model: nn.Module,                                                                                                                                            
     ...                                                                                                                                                                       
      20          config: TrainerConfig = TrainerConfig(),                                                                                                                     
      21          run: Optional[wandb.Run] = None,                                                                                                                             
      22      ) -> None:
      23 +        """Initialize the trainer with a model, optimizer, loss function, and optional W&B run.                                                               
      24 +                                                                                                                                                              
      25 +        Args:                                                                                                                                                 
      26 +            model: The neural network to train (moved to config.device).                                                                                      
      27 +            optimizer: The optimizer for updating model parameters.                                                                                           
      28 +            criterion: Loss function mapping (predictions, targets) -> scalar loss.                                                                           
      29 +            config: Training hyperparameters and device settings.                                                                                             
      30 +            run: Optional Weights & Biases run for experiment tracking.                                                                                       
      31 +        """                                                                                                                                                   
      32          self.model = model.to(config.device)
      33          self.optimizer = optimizer
      34          self.criterion = criterion

⏺ Update(src/my_engine/trainer.py)
  ⎿  Added 10 lines                                                                                                                                                            
      36          self.run = run                                                                                                                                               
      37  
      38      def train_one_epoch(self, train_loader) -> float:                                                                                                                
      39 +        """Run one training epoch over the entire train_loader.                                                                                                    
      40 +                                                                                                                                                                     
      41 +        Args:                                                                                                                                                      
      42 +            train_loader: DataLoader yielding (inputs, targets) batches.                                                                                             
      43 +                                                                                                                                                                     
      44 +        Returns:                                                                                                                                                     
      45 +            The sample-weighted average training loss for the epoch.                                                                                                 
      46 +        """                                                                                                                                                          
      47          self.model.train()
      48          total_loss = 0.0
      49          total_samples = 0
     ...
      51          for inputs, targets in train_loader:
      52              inputs, targets = inputs.to(self.config.device), targets.to(self.config.device)
      53  
      54 +            # Forward pass, backprop, and parameter update                                                                                                    
      55              self.optimizer.zero_grad()
      56              outputs = self.model(inputs)
      57              loss = self.criterion(outputs, targets)
      58              loss.backward()
      59              self.optimizer.step()
      60  
      61 +            # Accumulate weighted loss for averaging                                                                                                          
      62              batch_size = inputs.size(0)
      63              total_loss += loss.item() * batch_size
      64              total_samples += batch_size

⏺ Update(src/my_engine/trainer.py)
  ⎿  Added 9 lines                                                                                                                                                             
      70          return avg_loss                                                                                                                                              
      71  
      72      def validate(self, val_loader) -> Tuple[float, float]:                                                                                                           
      73 +        """Evaluate the model on a validation set without updating parameters.                                                                                     
      74 +                                                                                                                                                                     
      75 +        Args:                                                                                                                                                      
      76 +            val_loader: DataLoader yielding (inputs, targets) batches.                                                                                               
      77 +                                                                                                                                                                     
      78 +        Returns:                                                                                                                                                     
      79 +            A tuple of (average_loss, average_accuracy) over the validation set.                                                                                     
      80 +        """                                                                                                                                                          
      81          self.model.eval()
      82          running_loss = 0.0
      83          running_acc = 0.0
     ...
      92                  loss = self.criterion(logits, y_batch)
      93                  batch_acc = accuracy_from_logits(logits, y_batch)
      94  
      95 +                # Weight metrics by batch size for correct averaging                                                                                          
      96                  batch_size = X_batch.size(0)
      97                  running_loss += loss.item() * batch_size
      98                  running_acc += batch_acc * batch_size

⏺ Update(src/my_engine/trainer.py)
  ⎿  Added 11 lines                                                                                                                                                            
      106          return avg_loss, avg_acc                                                                                                                                    
      107  
      108      def fit(self, train_loader, val_loader) -> Dict[str, float]:                                                                                                    
      109 +        """Run the full training loop for config.num_epochs, validating after each epoch.                                                                         
      110                                                                                                                                                                      
      111 +        Args:                                                                                                                                                     
      112 +            train_loader: DataLoader for training data.                                                                                                             
      113 +            val_loader: DataLoader for validation data.                                                                                                             
      114 +                                                                                                                                                                    
      115 +        Returns:                                                                                                                                                    
      116 +            A dict with final metrics: num_epochs, train_loss, val_loss, val_acc.                                                                                   
      117 +                                                                                                                                                             
      118 +        Raises:                                                                                                                                              
      119 +            ValueError: If train_loader.batch_size doesn't match config.trainer_batch_size.                                                                  
      120 +        """                                                                                                                                                  
      121          # Sanity check: Verify the batch sizes match the config supplied
      122          if hasattr(train_loader, 'batch_size') and train_loader.batch_size is not None:
      123              if train_loader.batch_size != self.config.trainer_batch_size:

⏺ Done. Here's a summary of what was added:

  - Trainer class — one-line class docstring describing its purpose                                                                                                            
  - __init__ — docstring with full Args section for all parameters
  - train_one_epoch — docstring with Args/Returns, plus inline comments for the forward/backward pass and loss accumulation                                                    
  - validate — docstring with Args/Returns, plus a comment explaining the batch-weighted metric accumulation                                                                 
  - fit — docstring with Args/Returns/Raises sections

✻ Brewed for 40s                                                                                                                                                               
   
❯ do the same for the other files under my_engine                                                                                                                              
                                                                                                                                                                             
⏺ Searched for 1 pattern, read 4 files (ctrl+o to expand)                                                                                                                      
                                                                                                                                                                               
⏺ Now I'll add docstrings and comments to all three files.                                                                                                                     
                                                                                                                                                                               
⏺ Update(src/my_engine/config.py)                                                                                                                                              
  ⎿  Added 19 lines                                                                                                                                                            
      16                                                                                                                                                                       
      17  @dataclass                                                                                                                                                           
      18  class TrainerConfig:                                                                                                                                                 
      19 +    """Hyperparameters and settings for the training loop.                                                                                                         
      20 +                                                                                                                                                                     
      21 +    Attributes:                                                                                                                                                    
      22 +        trainer_batch_size: Batch size used during training.                                                                                                         
      23 +        evaluator_batch_size: Batch size used during evaluation/validation.                                                                                          
      24 +        learning_rate: Optimizer learning rate.                                                                                                                      
      25 +        device: Device to run training on (cpu or cuda).                                                                                                             
      26 +        num_epochs: Number of full passes over the training data.                                                                                                    
      27 +        weight_decay: L2 regularization coefficient for the optimizer.                                                                                        
      28 +        early_stopping_patience: Epochs to wait for improvement before stopping (None to disable).                                                            
      29 +        early_stopping_min_delta: Minimum change in loss to qualify as an improvement.                                                                        
      30 +    """                                                                                                                                                       
      31      trainer_batch_size: int = 64
      32      evaluator_batch_size: int = 256
      33      learning_rate: float = 0.001
     ...
      39  
      40  @dataclass
      41  class ModelConfig:
      42 +    """Architecture configuration for model construction.                                                                                                     
      43 +                                                                                                                                                              
      44 +    Attributes:                                                                                                                                               
      45 +        model_type: Model architecture identifier (e.g. "mlp").                                                                                               
      46 +        hidden_units: Number of neurons in each hidden layer.                                                                                                 
      47 +        dropout: Dropout rate after each hidden layer (aligned with hidden_units).                                                                            
      48 +    """                                                                                                                                                       
      49      model_type: str = "mlp"
      50      hidden_units: List[int] = field(default_factory=lambda: [128, 64])
      51      dropout: List[float] = field(default_factory=lambda: [0.1, 0.2])
                                                                                                                                                                               
⏺ Update(src/my_engine/model.py)       
  ⎿  Added 30 lines, removed 9 lines                                                                                                                                           
       4  from my_engine.config import ModelConfig                                                                                                                             
       5  
       6  class MLP_Model(nn.Module):                                                                                                                                          
       7 +    """A configurable multi-layer perceptron with ReLU activations and optional dropout."""                                                                        
       8 +                                                                                                                                                                     
       9      def __init__(self, num_inputs: int, num_outputs: int, config: ModelConfig):                                                                                    
      10 +        """Build the MLP layers from the given ModelConfig.                                                                                                          
      11 +                                                                                                                                                                     
      12 +        Args:                                                                                                                                                        
      13 +            num_inputs: Dimensionality of the input features.                                                                                                        
      14 +            num_outputs: Number of output classes/logits.                                                                                                            
      15 +            config: Specifies hidden layer sizes and dropout rates.                                                                                           
      16 +        """                                                                                                                                                   
      17          super().__init__()
       9 -        # DONE: Implement the initialization of the MLP model.                                                                                                
      18          self.num_inputs = num_inputs
      19          self.num_outputs = num_outputs
      20          self.config = config
      13 -                                                                                                                                                              
      21 +                                                                                                                                                              
      22          layers = []
      15 -                                                                                                                                                              
      23 +                                                                                                                                                              
      24 +        # Dynamically construct hidden layers with ReLU + optional dropout                                                                                    
      25          prev_dim = num_inputs
      26          for hidden_layer, dropout_rate in zip(config.hidden_units, config.dropout):
      27              layers.append(nn.Linear(prev_dim, hidden_layer))
      28              layers.append(nn.ReLU())
      29              if dropout_rate > 0:
      21 -                layers.append(nn.Dropout(dropout_rate))                                                                                                       
      30 +                layers.append(nn.Dropout(dropout_rate))                                                                                                       
      31              prev_dim = hidden_layer
      32 +        # Final projection to output logits (no activation)                                                                                                   
      33          layers.append(nn.Linear(prev_dim, num_outputs))
      24 -                                                                                                                                                              
      34 +                                                                                                                                                              
      35          self.layers = nn.Sequential(*layers)
      36  
      37      def forward(self, x: torch.Tensor) -> torch.Tensor:
      28 -        # DONE: Implement the forward pass of the MLP model.                                                                                                  
      38 +        """Pass input through all layers and return raw logits.                                                                                               
      39 +                                                                                                                                                              
      40 +        Args:                                                                                                                                                 
      41 +            x: Input tensor of shape (batch_size, num_inputs).                                                                                                
      42 +                                                                                                                                                              
      43 +        Returns:                                                                                                                                              
      44 +            Logits tensor of shape (batch_size, num_outputs).                                                                                                 
      45 +        """                                                                                                                                                   
      46          return self.layers(x)
      47  
      48      def num_parameters(self) -> tuple[int, int]:
      32 -        # DONE: Implement the number of parameters of the MLP model.                                                                                          
      49 +        """Count the total and trainable parameters in the model.                                                                                             
      50 +                                                                                                                                                              
      51 +        Returns:                                                                                                                                              
      52 +            A tuple of (total_parameters, trainable_parameters).                                                                                              
      53 +        """                                                                                                                                                   
      54          total = sum(p.numel() for p in self.parameters())
      55          trainable = sum(p.numel() for p in self.parameters() if p.requires_grad)
      56          return total, trainable
      57  
      58      def __str__(self) -> str:
      38 -        # DONE: Implement the string representation of the MLP model.                                                                                         
      59 +        """Return a human-readable summary of the model configuration."""                                                                                     
      60          return f"MLP_Model(num_inputs={self.num_inputs}, num_outputs={self.num_outputs}, config={self.config!r})"
      61  
      62      def __repr__(self) -> str:
      42 -        # DONE: Implement the representation of the MLP model.                                                                                                
      63 +        """Return the same string as __str__ for consistent display."""                                                                                       
      64          return str(self)

⏺ Update(src/my_engine/utils.py)        
  ⎿  Added 23 lines, removed 3 lines                                                                                                                                           
      18  from my_engine.model import MLP_Model                                                                                                                                
      19  
      20  def accuracy_from_logits(logits: torch.Tensor, labels: torch.Tensor) -> float:                                                                                       
      21 -    # DONE: Implement the accuracy_from_logits function.                                                                                                           
      21 +    """Compute classification accuracy from raw logits.                                                                                                              
      22 +                                                                                                                                                                   
      23 +    Args:                                                                                                                                                            
      24 +        logits: Model output of shape (batch_size, num_classes).                                                                                                     
      25 +        labels: Ground-truth class indices of shape (batch_size,).                                                                                                   
      26 +                                                                                                                                                                     
      27 +    Returns:                                                                                                                                                         
      28 +        Accuracy as a float in [0, 1].                                                                                                                        
      29 +    """                                                                                                                                                       
      30      predictions = torch.argmax(logits, dim=1)
      31      correct = (predictions == labels).sum().item()
      32      totals = labels.size(0)
      33      return correct / totals
      26 -                                                                                                                                                              
      34  
      35 +                                                                                                                                                              
      36  def build_model(num_inputs: int, num_outputs: int, config: ModelConfig) -> nn.Module:
      29 -    # DONE: Implement the build_model function.                                                                                                               
      37 +    """Factory function that constructs a model based on config.model_type.                                                                                   
      38 +                                                                                                                                                              
      39 +    Args:                                                                                                                                                     
      40 +        num_inputs: Dimensionality of input features.                                                                                                         
      41 +        num_outputs: Number of output classes.                                                                                                                
      42 +        config: Model architecture configuration.                                                                                                             
      43 +                                                                                                                                                              
      44 +    Returns:                                                                                                                                                  
      45 +        An instantiated nn.Module ready for training.                                                                                                         
      46 +                                                                                                                                                              
      47 +    Raises:                                                                                                                                                   
      48 +        ValueError: If config.model_type is not recognized.                                                                                                   
      49 +    """                                                                                                                                                       
      50      if config.model_type == "mlp":
      51          return MLP_Model(num_inputs, num_outputs, config)
      52      else:

⏺ Done. Here's what was added across the three files:

  config.py                                                                                                                                                                    
  - TrainerConfig — docstring listing all attributes and their purposes
  - ModelConfig — docstring listing all attributes                                                                                                                             
                                                                                                                                                                             
  model.py
  - MLP_Model class — one-line class docstring
  - __init__ — docstring with Args, inline comments for dynamic layer construction and output projection
  - forward — docstring with Args/Returns (input/output shapes)
  - num_parameters — docstring with Returns
  - __str__ / __repr__ — one-line docstrings
  - Removed the # DONE: comments throughout

  utils.py
  - accuracy_from_logits — docstring with Args/Returns
  - build_model — docstring with Args/Returns/Raises
  - Removed the # DONE: comments throughout
```
3.
```
okay, I fixed it by removing the !pip install -U where I was upgrading the colab packages. how does that work5:10 PMColab comes with a carefully pre-tested set of packages where all the compiled .so C extensions are built to work together. When you ran pip install -U, it upgraded individual packages (like scipy) to newer versions, but the compiled C extensions from the old version were still cached in memory (or partially overwritten on disk), creating a mismatch.
So what was happening:

Colab ships scipy 1.X with compiled .so files built for that exact version
pip install -U scipy upgrades the Python files to 1.17.0
But the old .so files (like _cyutility) are either still in memory or incompatible with the new Python files
Result: ImportError because the C extension doesn't export what the new Python code expects

By not upgrading, you just use Colab's pre-installed versions which are already compatible with each other. Colab's environment is essentially a fixed snapshot — it's designed to be used as-is, and upgrading individual packages breaks the internal consistency of that snapshot.
The general rule for Colab: only pip install packages that aren't already there, never -U upgrade existing ones unless you're prepared to restart the runtime fresh.wait, but I thought packages only get reloaded when we restart the runtime, no?5:11 PMYou're right — but the issue happens at the disk level, not just in memory.
When you run pip install -U scipy, it overwrites the files on disk immediately. So even though the old .so is still loaded in memory for the current session, the next time Python tries to import a different part of scipy that hasn't been loaded yet, it reads the new incompatible files from disk.
So the mismatch was:

Some scipy modules already loaded in memory → old version
New imports hitting disk → new version files
The two halves don't agree on the C extension API → ImportError

This is why it's such a messy error to debug — it's a partial upgrade mid-session, where part of scipy is the old version (in memory) and part is the new version (on disk).
```

# Clean shutdown for Colab

In [ ]:
# Disconnect the runtime if in Colab
if IN_COLAB:
    from google.colab import runtime
    print("Disconnecting runtime...")
    runtime.unassign()
else:
    print("Not running in Colab, skipping disconnect.")